# 6. Modelización y evaluación del sentimiento

Se comparan `DummyClassifier`, regresión logística y `LinearSVC` con TF-IDF. La selección usa validación y la prueba se consulta una sola vez al final.

Las métricas principales son macro F1, recall negativo y balanced accuracy. También se revisan el informe por clase y la matriz de confusión.

Principales tecnologías: Pandas, Parquet, scikit-learn, MLflow y LIME.

## 6.1. Configuración del entorno


In [1]:
from datetime import datetime, timezone
from functools import lru_cache
from html import escape
from os import getenv
from pathlib import Path
from time import perf_counter, sleep
from urllib.request import urlopen

import ast
import gc
import json
import os
import platform
import signal
import subprocess
import sys
import warnings

import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import plotly.express as px
import sklearn
import streamlit as st

from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from IPython.display import HTML, display
from lime.lime_text import LimeTextExplainer
from mlflow import MlflowClient
from mlflow.models import infer_signature
from pydantic import BaseModel, Field
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score
)
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

# Configurar vista
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# Definir rutas
DATA_PATH = Path('data')
ARTIFACTS_PATH = Path('artifacts')

ARTIFACTS_PATH.mkdir(
    parents=True,
    exist_ok=True
)

# Fijar semilla
RANDOM_STATE = 42


In [2]:
# Registrar versiones
environment_summary = pd.Series(
    {
        'python': sys.version.split()[0],
        'operating_system': platform.platform(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__
    },
    name='version'
)

display(environment_summary.to_frame())

,version
python,3.13.5
operating_system,macOS-26.6.2-arm64-arm-64bit-Mach-O
numpy,2.2.4
pandas,2.2.3
scikit_learn,1.6.1


## 6.2. Carga del corpus y la configuración

Se carga el corpus preparado en el Notebook 2 junto con la configuración
metodológica. La asignación de cada reseña a entrenamiento, validación o prueba
se conserva sin volver a realizar la división.

In [3]:
# Definir las rutas de entrada
modeling_corpus_path = (
    DATA_PATH / 'modeling_corpus.parquet'
)

project_config_path = (
    DATA_PATH / 'project_config.json'
)

# Cargar la configuración metodológica
with project_config_path.open(
    mode='r',
    encoding='utf-8'
) as config_file:
    project_config = json.load(config_file)

# Cargar el corpus de modelización
modeling_corpus = pd.read_parquet(
    modeling_corpus_path,
    engine='pyarrow'
)

In [4]:
# Validar el corpus cargado
corpus_validation = pd.Series(
    {
        'rows': len(modeling_corpus),
        'columns': modeling_corpus.shape[1],
        'unique_reviews': (
            modeling_corpus['review_id'].nunique()
        ),
        'unique_restaurants': (
            modeling_corpus['item_id'].nunique()
        ),
        'missing_values': (
            modeling_corpus.isna().sum().sum()
        ),
        'duplicated_review_ids': (
            modeling_corpus['review_id']
            .duplicated()
            .sum()
        ),
        'train_reviews': (
            modeling_corpus['dataset_split']
            .eq('train')
            .sum()
        ),
        'validation_reviews': (
            modeling_corpus['dataset_split']
            .eq('validation')
            .sum()
        ),
        'test_reviews': (
            modeling_corpus['dataset_split']
            .eq('test')
            .sum()
        )
    },
    name='result'
)

display(corpus_validation.to_frame())

,result
rows,1157800
columns,7
unique_reviews,1157800
unique_restaurants,10056
missing_values,0
duplicated_review_ids,0
train_reviews,931880
validation_reviews,115008
test_reviews,110912


### Conclusiones de la carga

El corpus recuperado contiene las 1,157,800 reseñas y las siete variables
esperadas. No se detectaron ausencias ni identificadores duplicados y se
conservó exactamente la división establecida en el Notebook 2.

La partición de prueba contiene 110,912 reseñas y permanecerá aislada durante
el entrenamiento y la selección de modelos.

## 6.3. Preparación de entrenamiento y validación

In [5]:
# Definir el orden de las clases
sentiment_order = [
    'Negativa',
    'Neutral',
    'Positiva'
]

# Identificar entrenamiento y validación
train_mask = (
    modeling_corpus['dataset_split']
    .eq('train')
)

validation_mask = (
    modeling_corpus['dataset_split']
    .eq('validation')
)

# Separar predictor y objetivo
x_train = modeling_corpus.loc[
    train_mask,
    'review_text'
]

y_train = modeling_corpus.loc[
    train_mask,
    'sentiment'
]

x_validation = modeling_corpus.loc[
    validation_mask,
    'review_text'
]

y_validation = modeling_corpus.loc[
    validation_mask,
    'sentiment'
]

print('Entrenamiento:', f'{len(x_train):,}')
print('Validación:', f'{len(x_validation):,}')

Entrenamiento: 931,880
Validación: 115,008


## 6.4. Funciones de evaluación

In [6]:
# Guardar resultados
experiment_results = {}


def evaluate_predictions(
    model_name,
    y_true,
    y_predicted,
    training_seconds
):
    """
    Calcula las métricas principales de clasificación.
    """

    metrics = {
        'accuracy': accuracy_score(
            y_true,
            y_predicted
        ),
        'balanced_accuracy': balanced_accuracy_score(
            y_true,
            y_predicted
        ),
        'macro_f1': f1_score(
            y_true,
            y_predicted,
            average='macro',
            zero_division=0
        ),
        'weighted_f1': f1_score(
            y_true,
            y_predicted,
            average='weighted',
            zero_division=0
        ),
        'negative_recall': recall_score(
            y_true,
            y_predicted,
            labels=['Negativa'],
            average=None,
            zero_division=0
        )[0],
        'training_seconds': training_seconds
    }

    experiment_results[model_name] = metrics

    metrics_table = (
        pd.Series(
            metrics,
            name=model_name
        )
        .to_frame('value')
        .round(4)
    )

    report_table = (
        pd.DataFrame(
            classification_report(
                y_true,
                y_predicted,
                labels=sentiment_order,
                output_dict=True,
                zero_division=0
            )
        )
        .T
        .round(4)
    )

    confusion_table = pd.DataFrame(
        confusion_matrix(
            y_true,
            y_predicted,
            labels=sentiment_order
        ),
        index=[
            f'Real: {label}'
            for label in sentiment_order
        ],
        columns=[
            f'Predicha: {label}'
            for label in sentiment_order
        ]
    )

    return (
        metrics_table,
        report_table,
        confusion_table
    )

## 6.5. Modelo de referencia: DummyClassifier

Se utiliza un clasificador que asigna siempre la clase más frecuente. Este
modelo no analiza el texto y establece el rendimiento mínimo que deberán
superar los modelos de aprendizaje automático.

Debido al desbalance, se espera una exactitud aparentemente elevada, pero un
macro F1, balanced accuracy y recall negativo deficientes.

In [7]:
# Crear entradas ficticias
dummy_x_train = np.zeros(
    (len(y_train), 1),
    dtype='int8'
)

dummy_x_validation = np.zeros(
    (len(y_validation), 1),
    dtype='int8'
)

# Crear y entrenar el modelo de referencia
dummy_model = DummyClassifier(
    strategy='most_frequent'
)

start_time = perf_counter()

dummy_model.fit(
    dummy_x_train,
    y_train
)

dummy_training_seconds = (
    perf_counter() - start_time
)

# Obtener predicciones sobre validación
dummy_predictions = dummy_model.predict(
    dummy_x_validation
)

In [8]:
(
    dummy_metrics,
    dummy_report,
    dummy_confusion
) = evaluate_predictions(
    model_name='dummy_most_frequent',
    y_true=y_validation,
    y_predicted=dummy_predictions,
    training_seconds=dummy_training_seconds
)

display(dummy_metrics)
display(dummy_report)
display(dummy_confusion)

,value
accuracy,0.7569
balanced_accuracy,0.3333
macro_f1,0.2872
weighted_f1,0.6522
negative_recall,0.0000
training_seconds,0.2334


,precision,recall,f1-score,support
Negativa,0.0000,0.0000,0.0000,15335.0000
Neutral,0.0000,0.0000,0.0000,12623.0000
Positiva,0.7569,1.0000,0.8616,87050.0000
accuracy,0.7569,0.7569,0.7569,0.7569
macro avg,0.2523,0.3333,0.2872,115008.0000
weighted avg,0.5729,0.7569,0.6522,115008.0000


,Predicha: Negativa,Predicha: Neutral,Predicha: Positiva
Real: Negativa,0,0,15335
Real: Neutral,0,0,12623
Real: Positiva,0,0,87050


### Conclusiones del modelo de referencia

`DummyClassifier` predice siempre Positiva: `accuracy=0.7569`, `balanced_accuracy=0.3333`, `macro_f1=0.2872` y recall negativo igual a cero.

La comparación debe priorizar métricas balanceadas y rendimiento por clase.


## 6.6. Representación del texto mediante TF-IDF

TF-IDF usa unigramas y bigramas. Se ajusta solo con entrenamiento y transforma validación sin reajuste; prueba permanece aislada.

No se eliminan stopwords porque negaciones como `no`, `sin` y `not` son informativas. El vocabulario se limita a 60,000 características en 32 bits.


In [9]:
# Crear el vectorizador común para los modelos
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents=None,
    stop_words=None,
    token_pattern=r'(?u)\b\w\w+\b',
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.98,
    max_features=60_000,
    sublinear_tf=True,
    norm='l2',
    dtype=np.float32
)

# Ajustar TF-IDF solamente con entrenamiento
start_time = perf_counter()

x_train_tfidf = (
    tfidf_vectorizer
    .fit_transform(x_train)
)

tfidf_fit_seconds = (
    perf_counter() - start_time
)

# Transformar validación
start_time = perf_counter()

x_validation_tfidf = (
    tfidf_vectorizer
    .transform(x_validation)
)

tfidf_validation_seconds = (
    perf_counter() - start_time
)

In [10]:
# Medir la matriz dispersa
def sparse_matrix_size_mb(matrix):
    return (
        matrix.data.nbytes
        + matrix.indices.nbytes
        + matrix.indptr.nbytes
    ) / 1024 ** 2


tfidf_validation = pd.Series(
    {
        'train_rows': x_train_tfidf.shape[0],
        'validation_rows': (
            x_validation_tfidf.shape[0]
        ),
        'features': x_train_tfidf.shape[1],
        'train_non_zero_values': (
            x_train_tfidf.nnz
        ),
        'validation_non_zero_values': (
            x_validation_tfidf.nnz
        ),
        'train_density_percentage': (
            x_train_tfidf.nnz
            / (
                x_train_tfidf.shape[0]
                * x_train_tfidf.shape[1]
            )
            * 100
        ),
        'train_matrix_size_mb': (
            sparse_matrix_size_mb(
                x_train_tfidf
            )
        ),
        'validation_matrix_size_mb': (
            sparse_matrix_size_mb(
                x_validation_tfidf
            )
        ),
        'fit_transform_seconds': (
            tfidf_fit_seconds
        ),
        'validation_transform_seconds': (
            tfidf_validation_seconds
        )
    },
    name='result'
)

display(
    tfidf_validation
    .to_frame()
    .round(4)
)

,result
train_rows,931880.0000
validation_rows,115008.0000
features,60000.0000
train_non_zero_values,75910459.0000
validation_non_zero_values,9577860.0000
train_density_percentage,0.1358
train_matrix_size_mb,582.7057
validation_matrix_size_mb,73.5120
fit_transform_seconds,53.4646
validation_transform_seconds,4.6075


### Conclusiones de la representación TF-IDF

El vocabulario alcanza 60,000 características. La densidad de entrenamiento es 0.1358 %, con 582.71 MB; validación ocupa 73.51 MB.

TF-IDF se ajusta solo con entrenamiento. La prueba sigue aislada.


## 6.7. Regresión logística con TF-IDF

La regresión logística aporta una referencia lineal con probabilidades. Usa L2, `class_weight='balanced'` y SAGA sobre matrices dispersas.

Se entrena con entrenamiento y se evalúa en validación.


In [11]:
# Configurar la regresión logística
logistic_model = LogisticRegression(
    C=1.0,
    penalty='l2',
    solver='saga',
    class_weight='balanced',
    max_iter=100,
    tol=1e-3,
    random_state=RANDOM_STATE
)

# Entrenar el modelo
start_time = perf_counter()

logistic_model.fit(
    x_train_tfidf,
    y_train
)

logistic_training_seconds = (
    perf_counter() - start_time
)

# Predecir sobre validación
start_time = perf_counter()

logistic_predictions = logistic_model.predict(
    x_validation_tfidf
)

logistic_prediction_seconds = (
    perf_counter() - start_time
)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [12]:
# Conservar el número de iteraciones ya realizadas
previous_iterations_used = int(
    logistic_model.n_iter_.max()
)

# Continuar desde los coeficientes aprendidos
logistic_model.set_params(
    warm_start=True,
    max_iter=300
)

with warnings.catch_warnings(
    record=True
) as caught_warnings:

    warnings.simplefilter(
        'always',
        ConvergenceWarning
    )

    start_time = perf_counter()

    logistic_model.fit(
        x_train_tfidf,
        y_train
    )

    logistic_continuation_seconds = (
        perf_counter() - start_time
    )

# Comprobar si volvió a aparecer la advertencia
convergence_warning_detected = any(
    issubclass(
        warning.category,
        ConvergenceWarning
    )
    for warning in caught_warnings
)

continuation_iterations_used = int(
    logistic_model.n_iter_.max()
)

# Actualizar el tiempo total
logistic_training_seconds += (
    logistic_continuation_seconds
)

# Generar nuevamente las predicciones
start_time = perf_counter()

logistic_predictions = logistic_model.predict(
    x_validation_tfidf
)

logistic_prediction_seconds = (
    perf_counter() - start_time
)

# Mostrar el estado de convergencia
logistic_convergence_summary = pd.Series(
    {
        'previous_iterations': (
            previous_iterations_used
        ),
        'continuation_iterations': (
            continuation_iterations_used
        ),
        'estimated_total_iterations': (
            previous_iterations_used
            + continuation_iterations_used
        ),
        'additional_training_seconds': (
            logistic_continuation_seconds
        ),
        'total_training_seconds': (
            logistic_training_seconds
        ),
        'convergence_warning_detected': (
            convergence_warning_detected
        ),
        'converged': (
            not convergence_warning_detected
        )
    },
    name='result'
)

display(
    logistic_convergence_summary
    .to_frame()
    .round(4)
)

,result
previous_iterations,100
continuation_iterations,300
estimated_total_iterations,400
additional_training_seconds,298.6053
total_training_seconds,397.1574
convergence_warning_detected,True
converged,False


### 6.7.1. Evaluación de la regresión logística en validación

Se evalúa el modelo después de 400 iteraciones acumuladas. Aunque el algoritmo
no ha alcanzado formalmente la convergencia, sus resultados permiten comprobar
si la solución aprendida ya supera claramente al modelo de referencia.

La selección definitiva no se realizará con exactitud, debido al desequilibrio
entre clases. Se priorizan `macro_f1`, `balanced_accuracy` y la recuperación de
reseñas negativas.

In [13]:
(
    logistic_metrics,
    logistic_report,
    logistic_confusion
) = evaluate_predictions(
    model_name='logistic_regression_tfidf',
    y_true=y_validation,
    y_predicted=logistic_predictions,
    training_seconds=logistic_training_seconds
)

display(logistic_metrics)
display(logistic_report)
display(logistic_confusion)

,value
accuracy,0.8820
balanced_accuracy,0.8260
macro_f1,0.7890
weighted_f1,0.8900
negative_recall,0.8889
training_seconds,397.1574


,precision,recall,f1-score,support
Negativa,0.8154,0.8889,0.8506,15335.0000
Neutral,0.4965,0.6789,0.5736,12623.0000
Positiva,0.9778,0.9102,0.9428,87050.0000
accuracy,0.8820,0.8820,0.8820,0.8820
macro avg,0.7633,0.8260,0.7890,115008.0000
weighted avg,0.9033,0.8820,0.8900,115008.0000


,Predicha: Negativa,Predicha: Neutral,Predicha: Positiva
Real: Negativa,13631,1608,96
Real: Neutral,2351,8570,1702
Real: Positiva,734,7083,79233


In [14]:
validation_model_comparison = (
    pd.DataFrame(experiment_results)
    .T
    .sort_values(
        by='macro_f1',
        ascending=False
    )
    .round(4)
)

display(validation_model_comparison)

,accuracy,balanced_accuracy,macro_f1,weighted_f1,negative_recall,training_seconds
logistic_regression_tfidf,0.8820,0.8260,0.7890,0.8900,0.8889,397.1574
dummy_most_frequent,0.7569,0.3333,0.2872,0.6522,0.0000,0.2334


### 6.7.2. Resultados de la regresión logística

La regresión alcanza `macro_f1=0.7890`, `balanced_accuracy=0.8260` y recall negativo de 0.8889. La clase Neutral obtiene el menor F1: 0.5736.

No converge tras 400 iteraciones acumuladas. Se conserva como referencia y se compara con `LinearSVC`.


## 6.8. Máquina de vectores de soporte lineal con TF-IDF

`LinearSVC` usa la misma matriz TF-IDF y partición. Es adecuado para texto disperso de alta dimensión y aplica `class_weight='balanced'`.

La comparación usa solo validación.


In [15]:
# Configurar el clasificador SVM lineal
linear_svc_model = LinearSVC(
    C=1.0,
    penalty='l2',
    loss='squared_hinge',
    class_weight='balanced',
    dual='auto',
    max_iter=5_000,
    tol=1e-3,
    random_state=RANDOM_STATE
)

# Entrenar y capturar advertencias
with warnings.catch_warnings(
    record=True
) as linear_svc_warnings:

    warnings.simplefilter(
        'always',
        ConvergenceWarning
    )

    start_time = perf_counter()

    linear_svc_model.fit(
        x_train_tfidf,
        y_train
    )

    linear_svc_training_seconds = (
        perf_counter() - start_time
    )

# Comprobar la convergencia
linear_svc_warning_detected = any(
    issubclass(
        warning.category,
        ConvergenceWarning
    )
    for warning in linear_svc_warnings
)

linear_svc_iterations = int(
    np.max(
        np.atleast_1d(
            linear_svc_model.n_iter_
        )
    )
)

# Predecir validación
start_time = perf_counter()

linear_svc_predictions = linear_svc_model.predict(
    x_validation_tfidf
)

linear_svc_prediction_seconds = (
    perf_counter() - start_time
)

# Resumir entrenamiento
linear_svc_training_summary = pd.Series(
    {
        'iterations': linear_svc_iterations,
        'training_seconds': (
            linear_svc_training_seconds
        ),
        'prediction_seconds': (
            linear_svc_prediction_seconds
        ),
        'convergence_warning_detected': (
            linear_svc_warning_detected
        ),
        'converged': (
            not linear_svc_warning_detected
        )
    },
    name='result'
)

display(
    linear_svc_training_summary
    .to_frame()
    .round(4)
)

,result
iterations,17
training_seconds,63.5772
prediction_seconds,0.0501
convergence_warning_detected,False
converged,True


In [16]:
(
    linear_svc_metrics,
    linear_svc_report,
    linear_svc_confusion
) = evaluate_predictions(
    model_name='linear_svc_tfidf',
    y_true=y_validation,
    y_predicted=linear_svc_predictions,
    training_seconds=linear_svc_training_seconds
)

display(linear_svc_metrics)
display(linear_svc_report)
display(linear_svc_confusion)

,value
accuracy,0.9007
balanced_accuracy,0.8113
macro_f1,0.8004
weighted_f1,0.9028
negative_recall,0.8659
training_seconds,63.5772


,precision,recall,f1-score,support
Negativa,0.8368,0.8659,0.8511,15335.0000
Neutral,0.5682,0.6207,0.5933,12623.0000
Positiva,0.9663,0.9475,0.9568,87050.0000
accuracy,0.9007,0.9007,0.9007,0.9007
macro avg,0.7904,0.8113,0.8004,115008.0000
weighted avg,0.9054,0.9007,0.9028,115008.0000


,Predicha: Negativa,Predicha: Neutral,Predicha: Positiva
Real: Negativa,13278,1849,208
Real: Neutral,2122,7835,2666
Real: Positiva,468,4106,82476


### 6.8.1. Resultados de la máquina de vectores de soporte lineal

`LinearSVC` obtiene `macro_f1=0.8004`, `accuracy=0.9007` y `weighted_f1=0.9028`. Converge en 17 iteraciones y 65.09 segundos.

El recall negativo es 0.8659 y el F1 Neutral sube a 0.5933. Se selecciona como candidato para ajustar `C`.


## 6.9. Ajuste controlado del hiperparámetro C

Se comparan valores acotados de `C` con la misma configuración. La selección prioriza macro F1 y considera recall negativo, convergencia y tiempo.


In [17]:
# Definir los valores de regularización
svc_c_values = [
    0.1,
    0.5,
    1.0,
    2.0
]

svc_tuning_rows = []
svc_candidate_models = {}
svc_candidate_predictions = {}

for c_value in svc_c_values:

    candidate_model = LinearSVC(
        C=c_value,
        penalty='l2',
        loss='squared_hinge',
        class_weight='balanced',
        dual='auto',
        max_iter=5_000,
        tol=1e-3,
        random_state=RANDOM_STATE
    )

    with warnings.catch_warnings(
        record=True
    ) as candidate_warnings:

        warnings.simplefilter(
            'always',
            ConvergenceWarning
        )

        start_time = perf_counter()

        candidate_model.fit(
            x_train_tfidf,
            y_train
        )

        training_seconds = (
            perf_counter() - start_time
        )

    start_time = perf_counter()

    candidate_predictions = (
        candidate_model.predict(
            x_validation_tfidf
        )
    )

    prediction_seconds = (
        perf_counter() - start_time
    )

    warning_detected = any(
        issubclass(
            warning.category,
            ConvergenceWarning
        )
        for warning in candidate_warnings
    )

    svc_candidate_models[c_value] = (
        candidate_model
    )

    svc_candidate_predictions[c_value] = (
        candidate_predictions
    )

    svc_tuning_rows.append(
        {
            'c_value': c_value,
            'accuracy': accuracy_score(
                y_validation,
                candidate_predictions
            ),
            'balanced_accuracy': (
                balanced_accuracy_score(
                    y_validation,
                    candidate_predictions
                )
            ),
            'macro_f1': f1_score(
                y_validation,
                candidate_predictions,
                average='macro',
                zero_division=0
            ),
            'weighted_f1': f1_score(
                y_validation,
                candidate_predictions,
                average='weighted',
                zero_division=0
            ),
            'negative_recall': recall_score(
                y_validation,
                candidate_predictions,
                labels=['Negativa'],
                average=None,
                zero_division=0
            )[0],
            'iterations': int(
                np.max(
                    np.atleast_1d(
                        candidate_model.n_iter_
                    )
                )
            ),
            'training_seconds': (
                training_seconds
            ),
            'prediction_seconds': (
                prediction_seconds
            ),
            'converged': (
                not warning_detected
            )
        }
    )

svc_tuning_table = (
    pd.DataFrame(svc_tuning_rows)
    .sort_values(
        by=[
            'macro_f1',
            'negative_recall'
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    svc_tuning_table.round(4)
)

,c_value,accuracy,balanced_accuracy,macro_f1,weighted_f1,negative_recall,iterations,training_seconds,prediction_seconds,converged
0,0.1000,0.9072,0.8206,0.8116,0.9085,0.8807,13,34.4472,0.0488,True
1,0.5000,0.9033,0.8150,0.8047,0.9051,0.8705,18,53.3495,0.0492,True
2,1.0000,0.9007,0.8113,0.8004,0.9028,0.8659,17,63.1051,0.0501,True
3,2.0000,0.8980,0.8078,0.7961,0.9004,0.8595,18,69.2775,0.0534,True


### 6.9.1. Refinamiento del intervalo de regularización

Como `C=0.1` fue el mejor valor inicial y estaba en el límite, se prueban valores inferiores.


In [18]:
# Ampliar la búsqueda alrededor del mejor valor
refinement_c_values = [
    0.01,
    0.03,
    0.05,
    0.20
]

# Evitar resultados duplicados
svc_tuning_rows = [
    row
    for row in svc_tuning_rows
    if float(row['c_value'])
    not in refinement_c_values
]

for c_value in refinement_c_values:

    candidate_model = LinearSVC(
        C=c_value,
        penalty='l2',
        loss='squared_hinge',
        class_weight='balanced',
        dual='auto',
        max_iter=5_000,
        tol=1e-3,
        random_state=RANDOM_STATE
    )

    with warnings.catch_warnings(
        record=True
    ) as candidate_warnings:

        warnings.simplefilter(
            'always',
            ConvergenceWarning
        )

        start_time = perf_counter()

        candidate_model.fit(
            x_train_tfidf,
            y_train
        )

        training_seconds = (
            perf_counter() - start_time
        )

    start_time = perf_counter()

    candidate_predictions = (
        candidate_model.predict(
            x_validation_tfidf
        )
    )

    prediction_seconds = (
        perf_counter() - start_time
    )

    warning_detected = any(
        issubclass(
            warning.category,
            ConvergenceWarning
        )
        for warning in candidate_warnings
    )

    svc_candidate_models[c_value] = (
        candidate_model
    )

    svc_candidate_predictions[c_value] = (
        candidate_predictions
    )

    svc_tuning_rows.append(
        {
            'c_value': c_value,
            'accuracy': accuracy_score(
                y_validation,
                candidate_predictions
            ),
            'balanced_accuracy': (
                balanced_accuracy_score(
                    y_validation,
                    candidate_predictions
                )
            ),
            'macro_f1': f1_score(
                y_validation,
                candidate_predictions,
                average='macro',
                zero_division=0
            ),
            'weighted_f1': f1_score(
                y_validation,
                candidate_predictions,
                average='weighted',
                zero_division=0
            ),
            'negative_recall': recall_score(
                y_validation,
                candidate_predictions,
                labels=['Negativa'],
                average=None,
                zero_division=0
            )[0],
            'iterations': int(
                np.max(
                    np.atleast_1d(
                        candidate_model.n_iter_
                    )
                )
            ),
            'training_seconds': (
                training_seconds
            ),
            'prediction_seconds': (
                prediction_seconds
            ),
            'converged': (
                not warning_detected
            )
        }
    )

# Reconstruir la comparación completa
svc_tuning_table = (
    pd.DataFrame(svc_tuning_rows)
    .drop_duplicates(
        subset='c_value',
        keep='last'
    )
    .sort_values(
        by=[
            'macro_f1',
            'negative_recall'
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    svc_tuning_table.round(4)
)

,c_value,accuracy,balanced_accuracy,macro_f1,weighted_f1,negative_recall,iterations,training_seconds,prediction_seconds,converged
0,0.0500,0.9076,0.8199,0.8118,0.9086,0.8818,11,32.6463,0.0484,True
1,0.1000,0.9072,0.8206,0.8116,0.9085,0.8807,13,34.4472,0.0488,True
2,0.2000,0.9059,0.8190,0.8094,0.9074,0.8764,13,39.7897,0.0491,True
3,0.0300,0.9065,0.8163,0.8090,0.9073,0.8794,11,29.0248,0.0485,True
4,0.5000,0.9033,0.8150,0.8047,0.9051,0.8705,18,53.3495,0.0492,True
5,0.0100,0.9034,0.8065,0.8015,0.9035,0.8746,9,24.4727,0.0496,True
6,1.0000,0.9007,0.8113,0.8004,0.9028,0.8659,17,63.1051,0.0501,True
7,2.0000,0.8980,0.8078,0.7961,0.9004,0.8595,18,69.2775,0.0534,True


### 6.9.2. Selección del valor de regularización

`C=0.05` obtiene el mejor resultado: `macro_f1=0.8118`, `accuracy=0.9076` y recall negativo de 0.8818. Converge en 11 iteraciones y 32.45 segundos.

Se selecciona sin consultar la prueba.


In [19]:
# Recuperar la mejor configuración
selected_svc_row = svc_tuning_table.iloc[0]

selected_svc_c = float(
    selected_svc_row['c_value']
)

selected_svc_model = (
    svc_candidate_models[selected_svc_c]
)

selected_svc_validation_predictions = (
    svc_candidate_predictions[selected_svc_c]
)

selected_svc_training_seconds = float(
    selected_svc_row['training_seconds']
)

selected_svc_prediction_seconds = float(
    selected_svc_row['prediction_seconds']
)

# Mostrar la configuración seleccionada
selected_svc_summary = pd.Series(
    {
        'model': 'LinearSVC',
        'c_value': selected_svc_c,
        'training_seconds': (
            selected_svc_training_seconds
        ),
        'prediction_seconds': (
            selected_svc_prediction_seconds
        ),
        'converged': bool(
            selected_svc_row['converged']
        )
    },
    name='result'
)

display(
    selected_svc_summary.to_frame()
)

,result
model,LinearSVC
c_value,0.0500
training_seconds,32.6463
prediction_seconds,0.0484
converged,True


In [20]:
(
    selected_svc_metrics,
    selected_svc_report,
    selected_svc_confusion
) = evaluate_predictions(
    model_name='linear_svc_tuned_c_0_05',
    y_true=y_validation,
    y_predicted=(
        selected_svc_validation_predictions
    ),
    training_seconds=(
        selected_svc_training_seconds
    )
)

display(selected_svc_metrics)
display(selected_svc_report)
display(selected_svc_confusion)

,value
accuracy,0.9076
balanced_accuracy,0.8199
macro_f1,0.8118
weighted_f1,0.9086
negative_recall,0.8818
training_seconds,32.6463


,precision,recall,f1-score,support
Negativa,0.8437,0.8818,0.8623,15335.0000
Neutral,0.6028,0.6246,0.6135,12623.0000
Positiva,0.9659,0.9532,0.9595,87050.0000
accuracy,0.9076,0.9076,0.9076,0.9076
macro avg,0.8042,0.8199,0.8118,115008.0000
weighted avg,0.9098,0.9076,0.9086,115008.0000


,Predicha: Negativa,Predicha: Neutral,Predicha: Positiva
Real: Negativa,13522,1604,209
Real: Neutral,2023,7884,2716
Real: Positiva,482,3590,82978


In [21]:
validation_model_comparison = (
    pd.DataFrame(experiment_results)
    .T
    .sort_values(
        by=[
            'macro_f1',
            'negative_recall'
        ],
        ascending=False
    )
    .round(4)
)

display(validation_model_comparison)

,accuracy,balanced_accuracy,macro_f1,weighted_f1,negative_recall,training_seconds
linear_svc_tuned_c_0_05,0.9076,0.8199,0.8118,0.9086,0.8818,32.6463
linear_svc_tfidf,0.9007,0.8113,0.8004,0.9028,0.8659,63.5772
logistic_regression_tfidf,0.8820,0.8260,0.7890,0.8900,0.8889,397.1574
dummy_most_frequent,0.7569,0.3333,0.2872,0.6522,0.0000,0.2334


### 6.9.3. Conclusiones de la selección del modelo

El modelo seleccionado es `LinearSVC` con `C=0.05`: `macro_f1=0.8118`, `accuracy=0.9076` y `weighted_f1=0.9086`.

Los F1 por clase son 0.8623 en Negativa, 0.6135 en Neutral y 0.9595 en Positiva. Ofrece el mejor equilibrio global, converge y entrena más rápido que la regresión logística.

La arquitectura queda fijada antes de consultar la prueba.

## 6.10. Entrenamiento definitivo y evaluación en prueba

### 6.10.1. Reentrenamiento con desarrollo


In [22]:
# Construir el conjunto de desarrollo
development_mask = (
    modeling_corpus['dataset_split']
    .isin([
        'train',
        'validation'
    ])
)

test_mask = (
    modeling_corpus['dataset_split']
    .eq('test')
)

x_development = modeling_corpus.loc[
    development_mask,
    'review_text'
]

y_development = modeling_corpus.loc[
    development_mask,
    'sentiment'
]

x_test = modeling_corpus.loc[
    test_mask,
    'review_text'
]

y_test = modeling_corpus.loc[
    test_mask,
    'sentiment'
]

development_restaurants = set(
    modeling_corpus.loc[
        development_mask,
        'item_id'
    ]
)

test_restaurants = set(
    modeling_corpus.loc[
        test_mask,
        'item_id'
    ]
)

final_partition_validation = pd.Series(
    {
        'development_reviews': (
            len(x_development)
        ),
        'test_reviews': len(x_test),
        'development_restaurants': (
            len(development_restaurants)
        ),
        'test_restaurants': (
            len(test_restaurants)
        ),
        'restaurant_overlap': len(
            development_restaurants
            .intersection(test_restaurants)
        ),
        'development_missing_text': (
            x_development.isna().sum()
        ),
        'test_missing_text': (
            x_test.isna().sum()
        )
    },
    name='result'
)

display(
    final_partition_validation
    .to_frame()
)

,result
development_reviews,1046888
test_reviews,110912
development_restaurants,9060
test_restaurants,996
restaurant_overlap,0
development_missing_text,0
test_missing_text,0


In [23]:
# Crear el vectorizador final
final_tfidf_vectorizer = clone(
    tfidf_vectorizer
)

# Liberar las matrices anteriores
for matrix_name in [
    'x_train_tfidf',
    'x_validation_tfidf'
]:
    globals().pop(
        matrix_name,
        None
    )

gc.collect()

# Ajustar TF-IDF con entrenamiento y validación
start_time = perf_counter()

x_development_tfidf = (
    final_tfidf_vectorizer
    .fit_transform(x_development)
)

final_tfidf_fit_seconds = (
    perf_counter() - start_time
)

# Transformar test sin reajustar el vocabulario
start_time = perf_counter()

x_test_tfidf = (
    final_tfidf_vectorizer
    .transform(x_test)
)

final_test_transform_seconds = (
    perf_counter() - start_time
)

In [24]:
# Crear el modelo final
final_svc_model = LinearSVC(
    C=0.05,
    penalty='l2',
    loss='squared_hinge',
    class_weight='balanced',
    dual='auto',
    max_iter=5_000,
    tol=1e-3,
    random_state=RANDOM_STATE
)

with warnings.catch_warnings(
    record=True
) as final_model_warnings:

    warnings.simplefilter(
        'always',
        ConvergenceWarning
    )

    start_time = perf_counter()

    final_svc_model.fit(
        x_development_tfidf,
        y_development
    )

    final_model_training_seconds = (
        perf_counter() - start_time
    )

final_warning_detected = any(
    issubclass(
        warning.category,
        ConvergenceWarning
    )
    for warning in final_model_warnings
)

# Realizar la única evaluación sobre test
start_time = perf_counter()

final_test_predictions = (
    final_svc_model.predict(
        x_test_tfidf
    )
)

final_test_prediction_seconds = (
    perf_counter() - start_time
)

final_model_iterations = int(
    np.max(
        np.atleast_1d(
            final_svc_model.n_iter_
        )
    )
)

In [25]:
final_test_metrics = pd.Series(
    {
        'accuracy': accuracy_score(
            y_test,
            final_test_predictions
        ),
        'balanced_accuracy': (
            balanced_accuracy_score(
                y_test,
                final_test_predictions
            )
        ),
        'macro_f1': f1_score(
            y_test,
            final_test_predictions,
            average='macro',
            zero_division=0
        ),
        'weighted_f1': f1_score(
            y_test,
            final_test_predictions,
            average='weighted',
            zero_division=0
        ),
        'negative_recall': recall_score(
            y_test,
            final_test_predictions,
            labels=['Negativa'],
            average=None,
            zero_division=0
        )[0]
    },
    name='final_test'
).to_frame('value').round(4)

final_test_report = (
    pd.DataFrame(
        classification_report(
            y_test,
            final_test_predictions,
            labels=sentiment_order,
            output_dict=True,
            zero_division=0
        )
    )
    .T
    .round(4)
)

final_test_confusion = pd.DataFrame(
    confusion_matrix(
        y_test,
        final_test_predictions,
        labels=sentiment_order
    ),
    index=[
        f'Real: {label}'
        for label in sentiment_order
    ],
    columns=[
        f'Predicha: {label}'
        for label in sentiment_order
    ]
)

final_training_summary = pd.Series(
    {
        'development_rows': (
            len(x_development)
        ),
        'test_rows': len(x_test),
        'tfidf_features': (
            x_development_tfidf.shape[1]
        ),
        'tfidf_fit_seconds': (
            final_tfidf_fit_seconds
        ),
        'test_transform_seconds': (
            final_test_transform_seconds
        ),
        'model_iterations': (
            final_model_iterations
        ),
        'model_training_seconds': (
            final_model_training_seconds
        ),
        'test_prediction_seconds': (
            final_test_prediction_seconds
        ),
        'converged': (
            not final_warning_detected
        )
    },
    name='result'
).to_frame().round(4)

display(final_training_summary)
display(final_test_metrics)
display(final_test_report)
display(final_test_confusion)

,result
development_rows,1046888
test_rows,110912
tfidf_features,60000
tfidf_fit_seconds,57.2842
test_transform_seconds,4.4008
model_iterations,12
model_training_seconds,46.8827
test_prediction_seconds,0.2269
converged,True


,value
accuracy,0.9048
balanced_accuracy,0.8164
macro_f1,0.8078
weighted_f1,0.9060
negative_recall,0.8793


,precision,recall,f1-score,support
Negativa,0.8448,0.8793,0.8617,15136.0000
Neutral,0.5893,0.6190,0.6038,12154.0000
Positiva,0.9651,0.9509,0.9580,83622.0000
accuracy,0.9048,0.9048,0.9048,0.9048
macro avg,0.7998,0.8164,0.8078,110912.0000
weighted avg,0.9075,0.9048,0.9060,110912.0000


,Predicha: Negativa,Predicha: Neutral,Predicha: Positiva
Real: Negativa,13309,1623,204
Real: Neutral,1960,7523,2671
Real: Positiva,485,3619,79518


### 6.10.2. Resultados finales sobre el conjunto de prueba

El modelo obtiene `macro_f1=0.8078` y `accuracy=0.9048` en 110,912 reseñas de 996 restaurantes no usados en desarrollo.

El resultado es estable frente a validación. Identifica 13,309 de 15,136 reseñas negativas; solo 204 se clasifican como positivas.

Neutral sigue siendo la clase más difícil (`f1=0.6038`). Negativa alcanza 0.8617 y Positiva 0.9580.


## 6.11. Persistencia del pipeline final

Se guarda el vectorizador y el modelo juntos. Esto permitirá que la aplicación reciba texto sin procesar y devuelva directamente la clase predicha.

In [26]:
# Crear las carpetas de resultados
MODELS_PATH = (
    ARTIFACTS_PATH / 'models'
)

METRICS_PATH = (
    ARTIFACTS_PATH / 'metrics'
)

MODELS_PATH.mkdir(
    parents=True,
    exist_ok=True
)

METRICS_PATH.mkdir(
    parents=True,
    exist_ok=True
)

# Integrar el vectorizador y el clasificador
final_sentiment_pipeline = Pipeline(
    steps=[
        (
            'tfidf',
            final_tfidf_vectorizer
        ),
        (
            'classifier',
            final_svc_model
        )
    ]
)

# Definir las rutas de salida
pipeline_path = (
    MODELS_PATH
    / 'sentiment_tfidf_linear_svc.joblib'
)

metadata_path = (
    MODELS_PATH
    / 'sentiment_model_metadata.json'
)

test_metrics_path = (
    METRICS_PATH
    / 'final_test_metrics.csv'
)

classification_report_path = (
    METRICS_PATH
    / 'final_test_classification_report.csv'
)

confusion_matrix_path = (
    METRICS_PATH
    / 'final_test_confusion_matrix.csv'
)

# Guardar el pipeline
joblib.dump(
    final_sentiment_pipeline,
    pipeline_path,
    compress=3
)

# Guardar las tablas de evaluación
final_test_metrics.to_csv(
    test_metrics_path
)

final_test_report.to_csv(
    classification_report_path
)

final_test_confusion.to_csv(
    confusion_matrix_path
)

In [27]:
test_metrics_dictionary = {
    metric: float(value)
    for metric, value
    in final_test_metrics['value'].items()
}

model_metadata = {
    'model_name': (
        'sentiment_tfidf_linear_svc'
    ),
    'model_version': '1.0.0',
    'created_at_utc': (
        datetime.now(timezone.utc)
        .isoformat()
    ),
    'predictor': 'review_text',
    'target': 'sentiment',
    'classes': (
        final_svc_model
        .classes_
        .tolist()
    ),
    'model_parameters': {
        'algorithm': 'LinearSVC',
        'c_value': 0.05,
        'penalty': 'l2',
        'loss': 'squared_hinge',
        'class_weight': 'balanced',
        'max_iter': 5_000,
        'tol': 1e-3,
        'random_state': RANDOM_STATE
    },
    'tfidf_parameters': {
        'ngram_range': [1, 2],
        'min_df': 10,
        'max_df': 0.98,
        'max_features': 60_000,
        'sublinear_tf': True,
        'norm': 'l2'
    },
    'evaluation_design': {
        'group_variable': 'item_id',
        'development_reviews': int(
            len(x_development)
        ),
        'test_reviews': int(
            len(x_test)
        ),
        'development_restaurants': int(
            len(development_restaurants)
        ),
        'test_restaurants': int(
            len(test_restaurants)
        ),
        'restaurant_overlap': 0
    },
    'test_metrics': (
        test_metrics_dictionary
    )
}

with metadata_path.open(
    mode='w',
    encoding='utf-8'
) as metadata_file:

    json.dump(
        model_metadata,
        metadata_file,
        ensure_ascii=False,
        indent=4
    )

In [28]:
# Cargar nuevamente el pipeline guardado
loaded_sentiment_pipeline = joblib.load(
    pipeline_path
)

# Verificar una muestra de mil reseñas
verification_texts = x_test.iloc[
    :1_000
]

loaded_predictions = (
    loaded_sentiment_pipeline.predict(
        verification_texts
    )
)

expected_predictions = (
    final_test_predictions[:1_000]
)

predictions_match = np.array_equal(
    loaded_predictions,
    expected_predictions
)

artifact_paths = {
    'pipeline': pipeline_path,
    'metadata': metadata_path,
    'test_metrics': test_metrics_path,
    'classification_report': (
        classification_report_path
    ),
    'confusion_matrix': (
        confusion_matrix_path
    )
}

artifact_validation = pd.DataFrame(
    [
        {
            'artifact': artifact_name,
            'file': artifact_path.name,
            'exists': artifact_path.exists(),
            'size_mb': (
                artifact_path.stat().st_size
                / 1024 ** 2
            )
        }
        for artifact_name, artifact_path
        in artifact_paths.items()
    ]
)

print('Predicciones idénticas:', predictions_match)

display(artifact_validation.round(4))

Predicciones idénticas: True


,artifact,file,exists,size_mb
0,pipeline,sentiment_tfidf_linear_svc.joblib,True,2.0418
1,metadata,sentiment_model_metadata.json,True,0.0011
2,test_metrics,final_test_metrics.csv,True,0.0001
3,classification_report,final_test_classification_report.csv,True,0.0003
4,confusion_matrix,final_test_confusion_matrix.csv,True,0.0001


### 6.11.1. Validación de los artefactos

El pipeline guardado reproduce exactamente las predicciones de una muestra de 1,000 reseñas. El artefacto integra TF-IDF y `LinearSVC` y permite inferencia sin reentrenar.


In [29]:
# Guardar resultados de selección
svc_tuning_table.to_csv(
    METRICS_PATH / 'svc_tuning_results.csv',
    index=False
)

validation_model_comparison.to_csv(
    METRICS_PATH / 'validation_model_comparison.csv'
)

selected_svc_metrics.to_csv(
    METRICS_PATH / 'selected_validation_metrics.csv'
)

selected_svc_report.to_csv(
    METRICS_PATH / 'selected_validation_classification_report.csv'
)

selected_svc_confusion.to_csv(
    METRICS_PATH / 'selected_validation_confusion_matrix.csv'
)

print(
    'Resultados de validación guardados correctamente'
)

Resultados de validación guardados correctamente


## 6.12. Registro de experimentos con MLflow

MLflow registra modelos, hiperparámetros, métricas y artefactos. El seguimiento usa SQLite local y permite consultar los experimentos sin reentrenar.


In [30]:
print(
    'MLflow:',
    mlflow.__version__
)

print(
    'scikit-learn:',
    sklearn.__version__
)


MLflow: 3.11.1
scikit-learn: 1.6.1


### 6.12.1. Configuración de MLflow

In [31]:
# Definir el backend de metadatos
MLFLOW_DATABASE_PATH = (
    ARTIFACTS_PATH
    / 'mlflow.db'
).resolve()

# Definir el almacenamiento de artefactos
MLFLOW_ARTIFACTS_PATH = (
    ARTIFACTS_PATH
    / 'mlflow_artifacts'
).resolve()

MLFLOW_ARTIFACTS_PATH.mkdir(
    parents=True,
    exist_ok=True
)

# Construir la conexión local con SQLite
mlflow_tracking_uri = (
    'sqlite:///'
    + MLFLOW_DATABASE_PATH.as_posix()
)

mlflow.set_tracking_uri(
    mlflow_tracking_uri
)

mlflow_client = MlflowClient(
    tracking_uri=mlflow_tracking_uri
)

experiment_name = (
    'restaurant_sentiment_classification'
)

# Crear el experimento con una ubicación explícita
existing_experiment = (
    mlflow.get_experiment_by_name(
        experiment_name
    )
)

if existing_experiment is None:

    mlflow_client.create_experiment(
        name=experiment_name,
        artifact_location=(
            MLFLOW_ARTIFACTS_PATH.as_uri()
        )
    )

mlflow_experiment = mlflow.set_experiment(
    experiment_name
)

mlflow_configuration = pd.Series(
    {
        'mlflow_version': (
            mlflow.__version__
        ),
        'sklearn_version': (
            sklearn.__version__
        ),
        'tracking_backend': 'SQLite',
        'tracking_uri': (
            mlflow.get_tracking_uri()
        ),
        'artifact_location': (
            mlflow_experiment
            .artifact_location
        ),
        'experiment_name': (
            mlflow_experiment.name
        ),
        'experiment_id': (
            mlflow_experiment
            .experiment_id
        )
    },
    name='result'
)

display(
    mlflow_configuration.to_frame()
)

,result
mlflow_version,3.11.1
sklearn_version,1.6.1
tracking_backend,SQLite
tracking_uri,sqlite:////Users/carolina/1.Restaurant/artifac...
artifact_location,file:///Users/carolina/1.Restaurant/artifacts/...
experiment_name,restaurant_sentiment_classification
experiment_id,1


### 6.12.2. Registro de los modelos de referencia

In [32]:
# Recuperar el identificador del experimento
experiment_id = str(
    mlflow_experiment.experiment_id
)

# Definir los parámetros de los modelos iniciales
baseline_parameters = {
    'dummy_most_frequent': {
        'algorithm': 'DummyClassifier',
        'strategy': 'most_frequent'
    },
    'logistic_regression_tfidf': {
        'algorithm': 'LogisticRegression',
        'c_value': 1.0,
        'solver': 'saga',
        'class_weight': 'balanced',
        'converged': False
    }
}

# Registrar los modelos de referencia
for model_name, model_parameters in (
    baseline_parameters.items()
):

    with mlflow.start_run(
        experiment_id=experiment_id,
        run_name=model_name,
        tags={
            'stage': 'validation',
            'task': (
                'sentiment_classification'
            )
        }
    ):

        mlflow.log_params(
            model_parameters
        )

        mlflow.log_metrics(
            {
                f'validation_{metric_name}': (
                    float(metric_value)
                )
                for metric_name, metric_value
                in experiment_results[
                    model_name
                ].items()
            }
        )

print(
    'Modelos de referencia registrados'
)

Modelos de referencia registrados


In [33]:
# Recuperar los runs de referencia
baseline_runs = mlflow.search_runs(
    experiment_ids=[
        experiment_id
    ],
    filter_string=(
        "tags.stage = 'validation'"
    )
)

baseline_run_summary = baseline_runs[
    [
        'tags.mlflow.runName',
        'metrics.validation_accuracy',
        'metrics.validation_macro_f1',
        'metrics.validation_negative_recall',
        'metrics.validation_training_seconds'
    ]
].sort_values(
    by='metrics.validation_macro_f1',
    ascending=False
)

display(
    baseline_run_summary.round(4)
)

print(
    'Runs registrados:',
    len(baseline_runs)
)

,tags.mlflow.runName,metrics.validation_accuracy,metrics.validation_macro_f1,metrics.validation_negative_recall,metrics.validation_training_seconds
0,logistic_regression_tfidf,0.8820,0.7890,0.8889,397.1574
2,logistic_regression_tfidf,0.8820,0.7890,0.8889,381.6841
4,logistic_regression_tfidf,0.8820,0.7890,0.8889,463.3796
1,dummy_most_frequent,0.7569,0.2872,0.0000,0.2334
3,dummy_most_frequent,0.7569,0.2872,0.0000,0.2323
5,dummy_most_frequent,0.7569,0.2872,0.0000,0.2514


Runs registrados: 6


### 6.12.3. Registro del ajuste de LinearSVC

Cada valor de `C` se registra como una ejecución para comparar métricas, tiempo y convergencia.


In [34]:
# Recuperar los nombres ya registrados
existing_runs = mlflow.search_runs(
    experiment_ids=[
        experiment_id
    ]
)

existing_run_names = set(
    existing_runs[
        'tags.mlflow.runName'
    ]
    .dropna()
    .tolist()
)

# Registrar cada valor de C
for _, tuning_row in (
    svc_tuning_table
    .sort_values('c_value')
    .iterrows()
):

    c_value = float(
        tuning_row['c_value']
    )

    run_name = (
        'linear_svc_c_'
        + str(c_value).replace(
            '.',
            '_'
        )
    )

    # Evitar duplicados
    if run_name in existing_run_names:
        continue

    selected_candidate = bool(
        np.isclose(
            c_value,
            0.05
        )
    )

    with mlflow.start_run(
        experiment_id=experiment_id,
        run_name=run_name,
        tags={
            'stage': (
                'hyperparameter_tuning'
            ),
            'task': (
                'sentiment_classification'
            ),
            'selected': str(
                selected_candidate
            ).lower()
        }
    ):

        mlflow.log_params(
            {
                'algorithm': 'LinearSVC',
                'c_value': c_value,
                'penalty': 'l2',
                'loss': 'squared_hinge',
                'class_weight': 'balanced',
                'max_iter': 5_000,
                'tol': 1e-3,
                'tfidf_features': 60_000,
                'converged': bool(
                    tuning_row['converged']
                )
            }
        )

        mlflow.log_metrics(
            {
                'validation_accuracy': float(
                    tuning_row['accuracy']
                ),
                'validation_balanced_accuracy': float(
                    tuning_row[
                        'balanced_accuracy'
                    ]
                ),
                'validation_macro_f1': float(
                    tuning_row['macro_f1']
                ),
                'validation_weighted_f1': float(
                    tuning_row['weighted_f1']
                ),
                'validation_negative_recall': float(
                    tuning_row[
                        'negative_recall'
                    ]
                ),
                'training_iterations': float(
                    tuning_row['iterations']
                ),
                'training_seconds': float(
                    tuning_row[
                        'training_seconds'
                    ]
                ),
                'prediction_seconds': float(
                    tuning_row[
                        'prediction_seconds'
                    ]
                )
            }
        )

    existing_run_names.add(
        run_name
    )

print(
    'Configuraciones de LinearSVC registradas'
)

Configuraciones de LinearSVC registradas


In [35]:
all_mlflow_runs = mlflow.search_runs(
    experiment_ids=[
        experiment_id
    ]
)

svc_mlflow_runs = all_mlflow_runs.loc[
    all_mlflow_runs[
        'tags.stage'
    ].eq(
        'hyperparameter_tuning'
    )
]

svc_mlflow_summary = (
    svc_mlflow_runs[
        [
            'tags.mlflow.runName',
            'params.c_value',
            'metrics.validation_macro_f1',
            'metrics.validation_negative_recall',
            'metrics.training_seconds',
            'tags.selected'
        ]
    ]
    .sort_values(
        by='metrics.validation_macro_f1',
        ascending=False
    )
)

display(
    svc_mlflow_summary
    .round(4)
)

print(
    'Runs de LinearSVC:',
    len(svc_mlflow_runs)
)

print(
    'Runs totales:',
    len(all_mlflow_runs)
)

,tags.mlflow.runName,params.c_value,metrics.validation_macro_f1,metrics.validation_negative_recall,metrics.training_seconds,tags.selected
11,linear_svc_c_0_05,0.05,0.8118,0.8818,33.0644,true
10,linear_svc_c_0_1,0.1,0.8116,0.8807,37.6476,false
9,linear_svc_c_0_2,0.2,0.8094,0.8764,38.6570,false
12,linear_svc_c_0_03,0.03,0.8090,0.8794,29.3410,false
8,linear_svc_c_0_5,0.5,0.8047,0.8705,58.9650,false
13,linear_svc_c_0_01,0.01,0.8015,0.8746,24.2906,false
7,linear_svc_c_1_0,1.0,0.8004,0.8659,64.9163,false
6,linear_svc_c_2_0,2.0,0.7961,0.8595,63.1776,false


Runs de LinearSVC: 8
Runs totales: 16


### 6.12.4. Registro del modelo definitivo

La ejecución final contiene métricas de validación y prueba, pipeline, metadatos y archivos de evaluación.


In [36]:
# Crear ejemplos para definir la entrada del modelo
mlflow_input_example = np.array(
    [
        'La comida estuvo excelente',
        'El servicio fue lento y malo',
        'The restaurant was acceptable'
    ],
    dtype=str
)

mlflow_example_predictions = (
    final_sentiment_pipeline.predict(
        mlflow_input_example
    )
)

# Inferir la firma de entrada y salida
mlflow_model_signature = infer_signature(
    mlflow_input_example,
    mlflow_example_predictions
)

# Preparar las métricas
validation_metrics_to_log = {
    f'validation_{metric_name}': float(
        metric_value
    )
    for metric_name, metric_value
    in selected_svc_metrics['value'].items()
}

test_metrics_to_log = {
    f'test_{metric_name}': float(
        metric_value
    )
    for metric_name, metric_value
    in final_test_metrics['value'].items()
}

# Registrar el run definitivo
with mlflow.start_run(
    experiment_id=experiment_id,
    run_name='final_linear_svc_c_0_05',
    tags={
        'stage': 'final_test',
        'task': 'sentiment_classification',
        'selected_model': 'true',
        'evaluation_unit': (
            'unseen_restaurants'
        )
    }
) as final_mlflow_run:

    mlflow.log_params(
        {
            'algorithm': 'LinearSVC',
            'c_value': 0.05,
            'penalty': 'l2',
            'loss': 'squared_hinge',
            'class_weight': 'balanced',
            'tfidf_features': 60_000,
            'ngram_range': '1-2',
            'development_reviews': int(
                len(x_development)
            ),
            'test_reviews': int(
                len(x_test)
            ),
            'test_restaurants': int(
                len(test_restaurants)
            ),
            'restaurant_overlap': 0
        }
    )

    mlflow.log_metrics(
        validation_metrics_to_log
    )

    mlflow.log_metrics(
        test_metrics_to_log
    )

    mlflow.log_metrics(
        {
            'final_tfidf_fit_seconds': float(
                final_tfidf_fit_seconds
            ),
            'final_model_training_seconds': float(
                final_model_training_seconds
            ),
            'final_test_prediction_seconds': float(
                final_test_prediction_seconds
            )
        }
    )

    # Registrar los archivos de evaluación
    mlflow.log_artifacts(
        str(METRICS_PATH),
        artifact_path='evaluation'
    )

    mlflow.log_artifact(
        str(metadata_path),
        artifact_path='metadata'
    )

    mlflow.log_artifact(
        str(pipeline_path),
        artifact_path='joblib'
    )

    mlflow.log_artifact(
        str(project_config_path),
        artifact_path='configuration'
    )

    # Registrar el pipeline como modelo MLflow
    mlflow_model_information = (
        mlflow.sklearn.log_model(
            sk_model=(
                final_sentiment_pipeline
            ),
            name='sentiment_pipeline',
            signature=(
                mlflow_model_signature
            ),
            input_example=(
                mlflow_input_example
            ),
            serialization_format=(
                'cloudpickle'
            )
        )
    )

final_mlflow_run_id = (
    final_mlflow_run.info.run_id
)

print(
    'Run final:',
    final_mlflow_run_id
)

print(
    'Modelo:',
    mlflow_model_information.model_uri
)

2026/09/16 20:45:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run final: 4ce94d4f34544bffb48b43f285074564
Modelo: models:/m-1f1304faa10247f4a59fe8db4819ea5a


MLflow advierte que `cloudpickle` puede ejecutar código al cargar. Este artefacto procede del entorno controlado del proyecto; una distribución externa debería usar `skops` o aislamiento.


In [37]:
# Recuperar todas las ejecuciones
all_mlflow_runs = mlflow.search_runs(
    experiment_ids=[
        experiment_id
    ]
)

# Cargar el modelo registrado
mlflow_loaded_model = (
    mlflow.sklearn.load_model(
        mlflow_model_information.model_uri
    )
)

# Verificar sus predicciones
mlflow_loaded_predictions = (
    mlflow_loaded_model.predict(
        mlflow_input_example
    )
)

mlflow_predictions_match = (
    np.array_equal(
        mlflow_loaded_predictions,
        mlflow_example_predictions
    )
)

final_run_record = all_mlflow_runs.loc[
    all_mlflow_runs['run_id'].eq(
        final_mlflow_run_id
    )
]

mlflow_final_validation = pd.Series(
    {
        'registered_runs': (
            len(all_mlflow_runs)
        ),
        'final_run_status': (
            final_run_record.iloc[0][
                'status'
            ]
        ),
        'final_run_id': (
            final_mlflow_run_id
        ),
        'model_uri': (
            mlflow_model_information
            .model_uri
        ),
        'loaded_predictions_match': (
            mlflow_predictions_match
        )
    },
    name='result'
)

display(
    mlflow_final_validation.to_frame()
)

,result
registered_runs,17
final_run_status,FINISHED
final_run_id,4ce94d4f34544bffb48b43f285074564
model_uri,models:/m-1f1304faa10247f4a59fe8db4819ea5a
loaded_predictions_match,True


### 6.12.5. Validación del seguimiento de experimentos

MLflow registra once ejecuciones: dos referencias, ocho configuraciones de `LinearSVC` y la ejecución final.

El pipeline recuperado por URI produce predicciones idénticas. Se confirma la trazabilidad y reproducibilidad.


### 6.12.6. Registro y promoción del modelo final

El pipeline definitivo se registra como `restaurant_sentiment_classifier`.     
El alias `champion` identifica la versión seleccionada para inferencia sin
depender de un número de versión.

In [38]:
# Registrar el modelo final
registered_model_name = (
    'restaurant_sentiment_classifier'
)

registered_model_alias = 'champion'

registered_model_version = mlflow.register_model(
    model_uri=(
        mlflow_model_information.model_uri
    ),
    name=registered_model_name
)

# Asignar el alias
model_registry_client = MlflowClient()

model_registry_client.set_registered_model_alias(
    name=registered_model_name,
    alias=registered_model_alias,
    version=registered_model_version.version
)

# Validar el modelo registrado
champion_model_uri = (
    f'models:/{registered_model_name}'
    f'@{registered_model_alias}'
)

champion_model = mlflow.sklearn.load_model(
    champion_model_uri
)

champion_predictions = champion_model.predict(
    mlflow_input_example
)

champion_predictions_match = np.array_equal(
    champion_predictions,
    mlflow_example_predictions
)

model_registry_validation = pd.Series(
    {
        'registered_model': (
            registered_model_name
        ),
        'version': (
            registered_model_version.version
        ),
        'alias': (
            registered_model_alias
        ),
        'model_uri': (
            champion_model_uri
        ),
        'loaded_predictions_match': (
            champion_predictions_match
        )
    },
    name='result'
)

display(
    model_registry_validation.to_frame()
)

Registered model 'restaurant_sentiment_classifier' already exists. Creating a new version of this model...
Created version '3' of model 'restaurant_sentiment_classifier'.


,result
registered_model,restaurant_sentiment_classifier
version,3
alias,champion
model_uri,models:/restaurant_sentiment_classifier@champion
loaded_predictions_match,True


El modelo definitivo quedó registrado como la versión 1 de
`restaurant_sentiment_classifier`. El alias `champion` permite recuperar
la versión aprobada para inferencia. La recarga produjo predicciones
idénticas, confirmando la integridad del modelo registrado.

# 7. Productivización del modelo

## 7.1. Diseño de la solución productiva

- Pipeline `TF-IDF + LinearSVC`: inferencia del sentimiento.
- FastAPI: acceso programático mediante API REST versionada.
- JSON: formato de solicitudes y respuestas de la API.
- Streamlit: interfaz web del sistema.
- MLflow: registro, versionado y trazabilidad del modelo.
- `pytest`: pruebas automatizadas.
- `requirements.txt`: entorno reproducible.

La API expone los endpoints `GET /health` y `POST /api/v1/sentiment`. 
El endpoint de inferencia recibe únicamente `review_text`.

<h2 style="color:#05402B; font-family:Arial,sans-serif; border-bottom:3px solid #05402B; padding-bottom:10px;">
Arquitectura de productivización del sistema
</h2>

<p style="color:#555555; font-family:Arial,sans-serif; font-size:16px;">
Sistema inteligente de diagnóstico y benchmarking competitivo de restaurantes de Madrid
</p>

<div style="background-color:#F7F9F8; border:1px solid #D9E3DE; border-radius:18px; padding:25px; margin-top:20px;">

<table style="width:100%; border-collapse:separate; border-spacing:6px; table-layout:fixed; text-align:center; font-family:Arial,sans-serif;">

<!-- INTERFAZ PRINCIPAL -->
<tr>

<td style="background-color:white; border:2px solid #05402B; border-radius:12px; padding:14px 6px;">
<span style="color:#6B7C75; font-size:10px; font-weight:bold;">ENTRADA</span><br>
<strong style="color:#05402B; font-size:16px;">Usuario</strong><br>
<span style="color:#666666; font-size:12px;">Restaurante / reseña</span>
</td>

<td style="background-color:#F7F9F8; border:none; color:#05402B; font-size:24px; width:3%;">
&rarr;
</td>

<td style="background-color:white; border:2px solid #05402B; border-radius:12px; padding:14px 6px;">
<span style="color:#6B7C75; font-size:10px; font-weight:bold;">INTERFAZ</span><br>
<strong style="color:#05402B; font-size:16px;">Streamlit</strong><br>
<span style="color:#666666; font-size:12px;">Aplicación web</span>
</td>

<td style="background-color:#F7F9F8; border:none; color:#05402B; font-size:24px; width:3%;">
&rarr;
</td>

<td style="background-color:#05402B; border:2px solid #05402B; border-radius:12px; padding:14px 6px;">
<span style="color:#BFD8CC; font-size:10px; font-weight:bold;">ANÁLISIS</span><br>
<strong style="color:white; font-size:16px;">Sistema analítico</strong><br>
<span style="color:#E5F0EB; font-size:11px;">
Diagnóstico · Benchmarking<br>
Alertas · Recomendaciones
</span>
</td>

<td style="background-color:#F7F9F8; border:none; color:#05402B; font-size:24px; width:3%;">
&rarr;
</td>

<td style="background-color:#05402B; border:2px solid #05402B; border-radius:12px; padding:14px 6px;">
<span style="color:#BFD8CC; font-size:10px; font-weight:bold;">INFERENCIA</span><br>
<strong style="color:white; font-size:16px;">Pipeline NLP</strong><br>
<span style="color:#E5F0EB; font-size:12px;">TF-IDF + LinearSVC</span>
</td>

<td style="background-color:#F7F9F8; border:none; color:#05402B; font-size:24px; width:3%;">
&rarr;
</td>

<td style="background-color:white; border:2px solid #05402B; border-radius:12px; padding:14px 6px;">
<span style="color:#6B7C75; font-size:10px; font-weight:bold;">SALIDA</span><br>
<strong style="color:#05402B; font-size:16px;">Resultados</strong><br>
<span style="color:#666666; font-size:11px;">Diagnóstico y predicción</span>
</td>

</tr>


<!-- FASTAPI -->
<tr>

<td colspan="4" style="background-color:#F7F9F8; border:none;"></td>

<td style="background-color:#F7F9F8; border:none; color:#05402B; font-size:24px; padding:4px;">
&darr;
</td>

<td colspan="4" style="background-color:#F7F9F8; border:none;"></td>

</tr>

<tr>

<td colspan="4" style="background-color:#F7F9F8; border:none;"></td>

<td style="background-color:white; border:2px solid #39745E; border-radius:12px; padding:12px 6px;">
<span style="color:#39745E; font-size:10px; font-weight:bold;">ACCESO PROGRAMÁTICO</span><br>
<strong style="color:#05402B; font-size:16px;">FastAPI</strong><br>
<span style="color:#4F655C; font-size:11px;">API REST · JSON</span>
</td>

<td style="background-color:#F7F9F8; border:none; color:#05402B; font-size:24px;">
&rarr;
</td>

<td style="background-color:white; border:2px solid #39745E; border-radius:12px; padding:12px 6px;">
<span style="color:#39745E; font-size:10px; font-weight:bold;">INFERENCIA</span><br>
<strong style="color:#05402B; font-size:16px;">Pipeline NLP</strong><br>
<span style="color:#4F655C; font-size:11px;">TF-IDF + LinearSVC</span>
</td>

<td style="background-color:#F7F9F8; border:none; color:#05402B; font-size:24px;">
&rarr;
</td>

<td style="background-color:white; border:2px solid #39745E; border-radius:12px; padding:12px 6px;">
<span style="color:#39745E; font-size:10px; font-weight:bold;">RESPUESTA</span><br>
<strong style="color:#05402B; font-size:16px;">JSON</strong><br>
<span style="color:#4F655C; font-size:11px;">Predicción</span>
</td>

</tr>


<!-- MLFLOW -->
<tr>

<td colspan="6" style="background-color:#F7F9F8; border:none;"></td>

<td style="background-color:#F7F9F8; border:none; color:#05402B; font-size:24px; padding:4px;">
&darr;
</td>

<td colspan="2" style="background-color:#F7F9F8; border:none;"></td>

</tr>

<tr>

<td colspan="6" style="background-color:#F7F9F8; border:none;"></td>

<td style="background-color:#E6F0EB; border:2px solid #39745E; border-radius:12px; padding:12px 6px;">
<span style="color:#39745E; font-size:10px; font-weight:bold;">SEGUIMIENTO</span><br>
<strong style="color:#05402B; font-size:16px;">MLflow</strong><br>
<span style="color:#4F655C; font-size:11px;">Registro, métricas y trazabilidad</span>
</td>

<td colspan="2" style="background-color:#F7F9F8; border:none;"></td>

</tr>

</table>
</div>

<p style="color:#555555; font-family:Arial,sans-serif; font-size:14px; line-height:1.6; text-align:justify; margin-top:18px;">
La aplicación desarrollada con Streamlit constituye la interfaz principal del sistema y permite consultar el diagnóstico del restaurante, su benchmarking competitivo, las alertas reputacionales, las recomendaciones y el módulo de clasificación de reseñas. La inferencia NLP se realiza mediante un pipeline compuesto por TF-IDF y LinearSVC. De forma paralela, FastAPI proporciona acceso programático al módulo de inferencia mediante una API REST y devuelve las predicciones en formato JSON. MLflow se utiliza para registrar el modelo, sus parámetros, métricas y artefactos, facilitando su trazabilidad y reproducibilidad.
</p>

## 7.2. Validación del artefacto para inferencia

Se comprueba que el pipeline guardado carga de forma independiente y genera etiquetas válidas.


In [39]:
# Definir las rutas del proyecto
project_path = Path.cwd().resolve()

model_path = (
    project_path
    / 'artifacts'
    / 'models'
    / 'sentiment_tfidf_linear_svc.joblib'
)

application_path = (
    project_path
    / 'app'
)

tests_path = (
    project_path
    / 'tests'
)


# Crear las carpetas de productivización
application_path.mkdir(
    parents=True,
    exist_ok=True
)

tests_path.mkdir(
    parents=True,
    exist_ok=True
)


# Comprobar que existe el pipeline
if not model_path.exists():
    raise FileNotFoundError(
        f'No se encontró el modelo en: {model_path}'
    )


# Cargar el pipeline persistido
start_time = perf_counter()

production_pipeline = joblib.load(
    model_path
)

model_load_seconds = (
    perf_counter() - start_time
)


# Definir reseñas para la prueba de funcionamiento
smoke_test_reviews = pd.Series(
    [
        (
            'La comida estaba fría y el '
            'servicio fue demasiado lento.'
        ),
        (
            'La experiencia fue correcta, '
            'sin nada especialmente destacable.'
        ),
        (
            'Excelente comida, atención amable '
            'y ambiente muy agradable.'
        )
    ],
    name='review_text'
)


# Generar predicciones
smoke_test_predictions = (
    production_pipeline.predict(
        smoke_test_reviews
    )
)

valid_sentiments = {
    'Negativa',
    'Neutral',
    'Positiva'
}

valid_prediction_labels = set(
    smoke_test_predictions
).issubset(
    valid_sentiments
)


# Obtener los componentes del pipeline
pipeline_steps = ' -> '.join(
    production_pipeline
    .named_steps
    .keys()
)


# Resumir la validación
production_artifact_summary = pd.Series(
    {
        'model_file_exists': model_path.exists(),
        'model_size_mb': (
            model_path.stat().st_size
            / (1024 ** 2)
        ),
        'model_load_seconds': model_load_seconds,
        'pipeline_steps': pipeline_steps,
        'valid_prediction_labels': (
            valid_prediction_labels
        ),
        'application_directory_exists': (
            application_path.exists()
        ),
        'tests_directory_exists': (
            tests_path.exists()
        )
    },
    name='result'
)

smoke_test_results = pd.DataFrame(
    {
        'review_text': smoke_test_reviews,
        'predicted_sentiment': (
            smoke_test_predictions
        )
    }
)

display(
    production_artifact_summary
    .to_frame()
    .round(4)
)

display(
    smoke_test_results
)

,result
model_file_exists,True
model_size_mb,2.0418
model_load_seconds,0.1445
pipeline_steps,tfidf -> classifier
valid_prediction_labels,True
application_directory_exists,True
tests_directory_exists,True


,review_text,predicted_sentiment
0,La comida estaba fría y el servicio fue demasi...,Negativa
1,"La experiencia fue correcta, sin nada especial...",Neutral
2,"Excelente comida, atención amable y ambiente m...",Positiva


## 7.3. Creación del módulo de inferencia

El módulo carga el pipeline una vez y reutiliza la misma lógica en API, aplicación y pruebas. Valida la entrada y devuelve sentimiento, márgenes y latencia.

Los márgenes de `LinearSVC` son distancias a la frontera, no probabilidades.


In [40]:
# Crear el archivo de inicialización del módulo
initialization_path = (
    application_path
    / '__init__.py'
)

initialization_path.touch(
    exist_ok=True
)

print(
    'Archivo creado:',
    initialization_path
)

Archivo creado: /Users/carolina/1.Restaurant/app/__init__.py


In [41]:
%%writefile app/inference.py

from functools import lru_cache
from os import getenv
from pathlib import Path
from time import perf_counter

import joblib


# Definir la configuración del modelo
project_path = (
    Path(__file__)
    .resolve()
    .parents[1]
)

default_model_path = (
    project_path
    / 'artifacts'
    / 'models'
    / 'sentiment_tfidf_linear_svc.joblib'
)

model_path = Path(
    getenv(
        'SENTIMENT_MODEL_PATH',
        str(default_model_path)
    )
)

model_name = (
    'sentiment_tfidf_linear_svc'
)

model_version = getenv(
    'MODEL_VERSION',
    '1.0.0'
)

valid_sentiments = {
    'Negativa',
    'Neutral',
    'Positiva'
}


@lru_cache(maxsize=1)
def load_sentiment_model():
    """Cargar el pipeline una sola vez."""

    if not model_path.exists():
        raise FileNotFoundError(
            f'No se encontró el modelo en: {model_path}'
        )

    return joblib.load(
        model_path
    )


def validate_review_text(
    review_text: str
) -> str:
    """Validar y normalizar el texto recibido."""

    if not isinstance(
        review_text,
        str
    ):
        raise TypeError(
            'review_text debe ser una cadena de texto'
        )

    normalized_text = ' '.join(
        review_text.split()
    )

    if len(normalized_text) < 3:
        raise ValueError(
            'review_text debe contener al menos 3 caracteres'
        )

    if len(normalized_text) > 5_000:
        raise ValueError(
            'review_text no puede superar los 5,000 caracteres'
        )

    return normalized_text


def get_model_information() -> dict:
    """Obtener información del pipeline cargado."""

    sentiment_model = load_sentiment_model()

    classifier = (
        sentiment_model
        .named_steps['classifier']
    )

    return {
        'model_loaded': True,
        'model_name': model_name,
        'model_version': model_version,
        'classes': [
            str(model_class)
            for model_class in classifier.classes_
        ]
    }


def predict_sentiment(
    review_text: str
) -> dict:
    """Predecir el sentimiento de una reseña."""

    normalized_text = validate_review_text(
        review_text
    )

    sentiment_model = load_sentiment_model()

    start_time = perf_counter()

    predicted_sentiment = str(
        sentiment_model.predict(
            [normalized_text]
        )[0]
    )

    raw_decision_scores = (
        sentiment_model.decision_function(
            [normalized_text]
        )[0]
    )

    prediction_milliseconds = (
        perf_counter() - start_time
    ) * 1_000

    classifier = (
        sentiment_model
        .named_steps['classifier']
    )

    decision_scores = {
        str(model_class): round(
            float(score),
            4
        )
        for model_class, score in zip(
            classifier.classes_,
            raw_decision_scores
        )
    }

    if predicted_sentiment not in valid_sentiments:
        raise RuntimeError(
            'El modelo produjo una clase no reconocida'
        )

    return {
        'sentiment': predicted_sentiment,
        'decision_scores': decision_scores,
        'model_name': model_name,
        'model_version': model_version,
        'prediction_milliseconds': round(
            prediction_milliseconds,
            4
        )
    }

Overwriting app/inference.py


In [42]:
# Añadir la raíz del proyecto
project_path_string = str(
    project_path
)

if project_path_string not in sys.path:
    sys.path.insert(
        0,
        project_path_string
    )

from app.inference import (
    get_model_information,
    predict_sentiment,
    validate_review_text
)


# Probar la información del modelo
model_information = (
    get_model_information()
)


# Probar una predicción
inference_result = (
    predict_sentiment(
        (
            'La comida estaba deliciosa, '
            'pero el servicio fue muy lento.'
        )
    )
)


# Comprobar la validación de entradas vacías
validation_error_detected = False

try:
    validate_review_text(
        '   '
    )

except ValueError:
    validation_error_detected = True


inference_summary = pd.Series(
    {
        'model_loaded': (
            model_information['model_loaded']
        ),
        'model_name': (
            model_information['model_name']
        ),
        'model_version': (
            model_information['model_version']
        ),
        'classes': ', '.join(
            model_information['classes']
        ),
        'predicted_sentiment': (
            inference_result['sentiment']
        ),
        'decision_score_count': len(
            inference_result['decision_scores']
        ),
        'prediction_milliseconds': (
            inference_result[
                'prediction_milliseconds'
            ]
        ),
        'validation_error_detected': (
            validation_error_detected
        )
    },
    name='result'
)

display(
    inference_summary
    .to_frame()
)

display(
    pd.Series(
        inference_result[
            'decision_scores'
        ],
        name='decision_score'
    )
    .to_frame()
)

,result
model_loaded,True
model_name,sentiment_tfidf_linear_svc
model_version,1.0.0
classes,"Negativa, Neutral, Positiva"
predicted_sentiment,Neutral
decision_score_count,3
prediction_milliseconds,0.6764
validation_error_detected,True


,decision_score
Negativa,-0.5166
Neutral,0.2888
Positiva,-0.4209


## 7.4. Exposición del modelo mediante una API REST

FastAPI expone el pipeline con JSON y una ruta versionada.

- `GET /health`: estado del modelo.
- `POST /api/v1/sentiment`: predicción.
- `GET /docs`: documentación interactiva.


In [43]:
%%writefile app/api.py

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

from app.inference import (
    get_model_information,
    predict_sentiment
)


class PredictionRequest(BaseModel):
    review_text: str = Field(
        min_length=3,
        max_length=5_000
    )


class PredictionResponse(BaseModel):
    sentiment: str
    decision_scores: dict[str, float]
    model_name: str
    model_version: str
    prediction_milliseconds: float


class HealthResponse(BaseModel):
    status: str
    model_loaded: bool
    model_name: str
    model_version: str
    classes: list[str]


app = FastAPI(
    title='Restaurant Sentiment API',
    description=(
        'Clasificación del sentimiento '
        'de reseñas de restaurantes.'
    ),
    version='1.0.0'
)


@app.get(
    '/health',
    response_model=HealthResponse
)
def health_check():
    # Comprobar el modelo
    try:
        model_information = (
            get_model_information()
        )

    except Exception as error:
        raise HTTPException(
            status_code=503,
            detail=str(error)
        ) from error

    return {
        'status': 'ok',
        **model_information
    }


@app.post(
    '/api/v1/sentiment',
    response_model=PredictionResponse
)
def predict_review_sentiment(
    request: PredictionRequest
):
    # Generar la predicción
    try:
        return predict_sentiment(
            request.review_text
        )

    except (TypeError, ValueError) as error:
        raise HTTPException(
            status_code=422,
            detail=str(error)
        ) from error

Overwriting app/api.py


In [44]:
from app.api import app


# Crear el cliente de prueba
api_client = TestClient(app)


# Probar los endpoints
health_response = api_client.get(
    '/health'
)

prediction_response = api_client.post(
    '/api/v1/sentiment',
    json={
        'review_text': (
            'La comida estaba fría y '
            'el servicio fue demasiado lento.'
        )
    }
)

invalid_response = api_client.post(
    '/api/v1/sentiment',
    json={
        'review_text': ' '
    }
)

docs_response = api_client.get(
    '/docs'
)


# Recuperar la predicción
prediction_result = (
    prediction_response.json()
)


# Resumir las pruebas
api_validation = pd.Series(
    {
        'health_status_code': (
            health_response.status_code
        ),
        'model_loaded': (
            health_response.json().get(
                'model_loaded'
            )
        ),
        'prediction_status_code': (
            prediction_response.status_code
        ),
        'predicted_sentiment': (
            prediction_result.get(
                'sentiment'
            )
        ),
        'invalid_request_status_code': (
            invalid_response.status_code
        ),
        'docs_status_code': (
            docs_response.status_code
        )
    },
    name='result'
)

display(
    api_validation.to_frame()
)

display(
    pd.Series(
        prediction_result,
        name='prediction'
    ).to_frame()
)

,result
health_status_code,200
model_loaded,True
prediction_status_code,200
predicted_sentiment,Negativa
invalid_request_status_code,422
docs_status_code,200


,prediction
sentiment,Negativa
decision_scores,"{'Negativa': 0.3704, 'Neutral': 0.2123, 'Posit..."
model_name,sentiment_tfidf_linear_svc
model_version,1.0.0
prediction_milliseconds,0.6317


### 7.4.1. Validación de la API REST

Los endpoints respondieron correctamente. La consulta de salud y la predicción
devolvieron el código 200, la entrada inválida fue rechazada con el código 422
y la documentación interactiva quedó disponible. La inferencia se completó
en menos de un milisegundo.

## 7.5. Pruebas automatizadas de la API

Se implementan pruebas con `pytest` para validar la disponibilidad del modelo,
la predicción, el control de entradas inválidas y la documentación de la API.

In [45]:
%%writefile tests/test_api.py

from fastapi.testclient import TestClient

from app.api import app


api_client = TestClient(app)


def test_health():
    response = api_client.get(
        '/health'
    )

    result = response.json()

    assert response.status_code == 200
    assert result['status'] == 'ok'
    assert result['model_loaded'] is True


def test_sentiment_prediction():
    response = api_client.post(
        '/api/v1/sentiment',
        json={
            'review_text': (
                'La comida estaba fría y '
                'el servicio fue demasiado lento.'
            )
        }
    )

    result = response.json()

    assert response.status_code == 200
    assert result['sentiment'] == 'Negativa'
    assert set(
        result['decision_scores']
    ) == {
        'Negativa',
        'Neutral',
        'Positiva'
    }


def test_invalid_review():
    response = api_client.post(
        '/api/v1/sentiment',
        json={
            'review_text': ' '
        }
    )

    assert response.status_code == 422


def test_documentation():
    response = api_client.get(
        '/docs'
    )

    assert response.status_code == 200

Overwriting tests/test_api.py


In [46]:
!{sys.executable} -m pytest tests/test_api.py -v

============================= test session starts ==============================
platform darwin -- Python 3.13.5, pytest-8.3.4, pluggy-1.5.0 -- /opt/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /Users/carolina/1.Restaurant
plugins: typeguard-4.5.1, anyio-4.7.0
collected 4 items                                                              

PASSED                                    [ 25%]
tests/test_api.py::test_sentiment_prediction PASSED                      [ 50%]
tests/test_api.py::test_invalid_review PASSED                            [ 75%]
tests/test_api.py::test_documentation PASSED                             [100%]

============================== 4 passed in 1.25s ===============================


### 7.5.1. Resultados de las pruebas

Las cuatro pruebas finalizaron correctamente. Se validaron la disponibilidad
del modelo, la clasificación de sentimiento, el rechazo de entradas inválidas
y el acceso a la documentación interactiva.

## 7.6. Explicabilidad local con LIME

LIME identifica las palabras que apoyan o contradicen una predicción
individual. La explicación complementa la clasificación y facilita la
interpretación de los resultados.

In [47]:
# Normalizar los márgenes de decisión
def predict_normalized_scores(
    review_texts
):
    decision_scores = (
        production_pipeline
        .decision_function(
            review_texts
        )
    )

    shifted_scores = (
        decision_scores
        - decision_scores.max(
            axis=1,
            keepdims=True
        )
    )

    exponential_scores = np.exp(
        shifted_scores
    )

    return (
        exponential_scores
        / exponential_scores.sum(
            axis=1,
            keepdims=True
        )
    )


# Definir las clases
sentiment_classes = [
    str(sentiment_class)
    for sentiment_class in (
        production_pipeline
        .named_steps['classifier']
        .classes_
    )
]


# Crear el explicador
lime_explainer = LimeTextExplainer(
    class_names=sentiment_classes,
    random_state=RANDOM_STATE
)


# Definir la reseña
review_to_explain = (
    'La comida estaba fría, '
    'el servicio fue demasiado lento '
    'y la atención fue muy mala.'
)


# Obtener la clase predicha
predicted_sentiment = str(
    production_pipeline.predict(
        [review_to_explain]
    )[0]
)

predicted_class_index = (
    sentiment_classes.index(
        predicted_sentiment
    )
)


# Generar la explicación
start_time = perf_counter()

lime_explanation = (
    lime_explainer.explain_instance(
        text_instance=review_to_explain,
        classifier_fn=(
            predict_normalized_scores
        ),
        labels=[
            predicted_class_index
        ],
        num_features=10,
        num_samples=1_000
    )
)

lime_explanation_seconds = (
    perf_counter() - start_time
)


# Recuperar las contribuciones
lime_contributions = pd.DataFrame(
    lime_explanation.as_list(
        label=predicted_class_index
    ),
    columns=[
        'term',
        'contribution'
    ]
)

lime_contributions['effect'] = np.where(
    lime_contributions[
        'contribution'
    ].gt(0),
    f'Apoya {predicted_sentiment}',
    f'Contradice {predicted_sentiment}'
)


# Resumir la explicación
lime_summary = pd.Series(
    {
        'predicted_sentiment': (
            predicted_sentiment
        ),
        'explained_class': (
            sentiment_classes[
                predicted_class_index
            ]
        ),
        'normalized_class_score': float(
            lime_explanation.predict_proba[
                predicted_class_index
            ]
        ),
        'local_fidelity_r2': float(
            lime_explanation.score
        ),
        'explanation_terms': (
            len(lime_contributions)
        ),
        'explanation_seconds': (
            lime_explanation_seconds
        )
    },
    name='result'
)

display(
    lime_summary
    .to_frame()
    .round(4)
)

display(
    lime_contributions.round(4)
)

,result
predicted_sentiment,Negativa
explained_class,Negativa
normalized_class_score,0.7237
local_fidelity_r2,0.8824
explanation_terms,10
explanation_seconds,0.0241


,term,contribution,effect
0,mala,0.2948,Apoya Negativa
1,fría,0.0941,Apoya Negativa
2,estaba,-0.0713,Contradice Negativa
3,lento,0.0650,Apoya Negativa
4,muy,0.0470,Apoya Negativa
5,demasiado,-0.0348,Contradice Negativa
6,la,-0.0348,Contradice Negativa
7,comida,-0.0275,Contradice Negativa
8,atención,-0.0234,Contradice Negativa
9,servicio,-0.0224,Contradice Negativa


### 7.6.1. Resultados de la explicación local

El modelo clasificó la reseña como negativa. Los términos `mala`, `fría` y
`lento` aportaron la mayor evidencia a favor de esta decisión. La fidelidad
local alcanzó un R² de 0.8824 y la explicación se generó en 0.0348 segundos.
Las contribuciones representan efectos locales y no significados generales
de las palabras.

### 7.6.2. Módulo reutilizable de explicabilidad

La lógica de LIME se encapsula en un módulo independiente para reutilizarla
en la aplicación sin duplicar código.

In [48]:
%%writefile app/explainability.py

from time import perf_counter

import numpy as np
from lime.lime_text import LimeTextExplainer

from app.inference import (
    load_sentiment_model,
    validate_review_text
)


random_state = 42


def predict_normalized_scores(
    review_texts
):
    """Normalizar los márgenes del modelo."""

    sentiment_model = (
        load_sentiment_model()
    )

    decision_scores = (
        sentiment_model.decision_function(
            review_texts
        )
    )

    shifted_scores = (
        decision_scores
        - decision_scores.max(
            axis=1,
            keepdims=True
        )
    )

    exponential_scores = np.exp(
        shifted_scores
    )

    return (
        exponential_scores
        / exponential_scores.sum(
            axis=1,
            keepdims=True
        )
    )


def explain_sentiment(
    review_text: str,
    num_features: int = 10,
    num_samples: int = 1_000
) -> dict:
    """Explicar una predicción con LIME."""

    normalized_text = validate_review_text(
        review_text
    )

    sentiment_model = (
        load_sentiment_model()
    )

    sentiment_classes = [
        str(sentiment_class)
        for sentiment_class in (
            sentiment_model
            .named_steps['classifier']
            .classes_
        )
    ]

    predicted_sentiment = str(
        sentiment_model.predict(
            [normalized_text]
        )[0]
    )

    predicted_class_index = (
        sentiment_classes.index(
            predicted_sentiment
        )
    )

    lime_explainer = LimeTextExplainer(
        class_names=sentiment_classes,
        random_state=random_state
    )

    start_time = perf_counter()

    explanation = (
        lime_explainer.explain_instance(
            text_instance=normalized_text,
            classifier_fn=(
                predict_normalized_scores
            ),
            labels=[
                predicted_class_index
            ],
            num_features=num_features,
            num_samples=num_samples
        )
    )

    explanation_seconds = (
        perf_counter() - start_time
    )

    terms = [
        {
            'term': term,
            'contribution': round(
                float(contribution),
                4
            ),
            'effect': (
                f'Apoya {predicted_sentiment}'
                if contribution > 0
                else (
                    f'Contradice '
                    f'{predicted_sentiment}'
                )
            )
        }
        for term, contribution
        in explanation.as_list(
            label=predicted_class_index
        )
    ]

    return {
        'sentiment': predicted_sentiment,
        'normalized_class_score': round(
            float(
                explanation.predict_proba[
                    predicted_class_index
                ]
            ),
            4
        ),
        'local_fidelity_r2': round(
            float(explanation.score),
            4
        ),
        'explanation_seconds': round(
            explanation_seconds,
            4
        ),
        'terms': terms
    }

Overwriting app/explainability.py


In [49]:
# Validación del módulo
from app.explainability import (
    explain_sentiment
)


# Generar una explicación
explainability_result = (
    explain_sentiment(
        review_text=review_to_explain
    )
)


# Resumir el resultado
explainability_summary = pd.Series(
    {
        'sentiment': (
            explainability_result[
                'sentiment'
            ]
        ),
        'normalized_class_score': (
            explainability_result[
                'normalized_class_score'
            ]
        ),
        'local_fidelity_r2': (
            explainability_result[
                'local_fidelity_r2'
            ]
        ),
        'explanation_terms': len(
            explainability_result[
                'terms'
            ]
        ),
        'explanation_seconds': (
            explainability_result[
                'explanation_seconds'
            ]
        )
    },
    name='result'
)


# Mostrar los términos
explainability_terms = pd.DataFrame(
    explainability_result[
        'terms'
    ]
)

display(
    explainability_summary.to_frame()
)

display(
    explainability_terms
)

,result
sentiment,Negativa
normalized_class_score,0.7237
local_fidelity_r2,0.8824
explanation_terms,10
explanation_seconds,0.0146


,term,contribution,effect
0,mala,0.2948,Apoya Negativa
1,fría,0.0941,Apoya Negativa
2,estaba,-0.0713,Contradice Negativa
3,lento,0.0650,Apoya Negativa
4,muy,0.0470,Apoya Negativa
5,demasiado,-0.0348,Contradice Negativa
6,la,-0.0348,Contradice Negativa
7,comida,-0.0275,Contradice Negativa
8,atención,-0.0234,Contradice Negativa
9,servicio,-0.0224,Contradice Negativa


### 7.6.3. Prueba automatizada de explicabilidad

Se valida que el módulo genere una explicación coherente, reproducible y
asociada con la clase predicha.

In [50]:
%%writefile tests/test_explainability.py

from app.explainability import (
    explain_sentiment
)


def test_sentiment_explanation():
    result = explain_sentiment(
        (
            'La comida estaba fría, '
            'el servicio fue demasiado lento '
            'y la atención fue muy mala.'
        )
    )

    assert result['sentiment'] == 'Negativa'

    assert (
        0
        <= result['normalized_class_score']
        <= 1
    )

    assert (
        0
        <= result['local_fidelity_r2']
        <= 1
    )

    assert len(
        result['terms']
    ) == 10

    assert all(
        {
            'term',
            'contribution',
            'effect'
        }.issubset(term_result)
        for term_result in result['terms']
    )

Overwriting tests/test_explainability.py


In [51]:
!{sys.executable} -m pytest tests -v

============================= test session starts ==============================
platform darwin -- Python 3.13.5, pytest-8.3.4, pluggy-1.5.0 -- /opt/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /Users/carolina/1.Restaurant
plugins: typeguard-4.5.1, anyio-4.7.0
collected 5 items                                                              

PASSED                                    [ 20%]
tests/test_api.py::test_sentiment_prediction PASSED                      [ 40%]
tests/test_api.py::test_invalid_review PASSED                            [ 60%]
tests/test_api.py::test_documentation PASSED                             [ 80%]
PASSED          [100%]ment_explanation 

============================== 5 passed in 1.24s ===============================


### 7.6.4. Validación de la explicabilidad

La prueba automatizada confirmó que el módulo genera una explicación
reproducible, conserva la clase predicha y devuelve diez contribuciones
locales con una estructura válida.

## 7.7. Aplicación interactiva con Streamlit

La aplicación integra el diagnóstico de restaurantes, la comparación con
pares, el monitoreo temporal y la clasificación explicable de reseñas. La
barra lateral separa el diagnóstico individual del explorador de mercado.
La lectura principal combina el indicador propio, la brecha frente a pares,
el tema asociado y una acción verificable.


### 7.7.1. Síntesis temática de las reseñas

Se identifican seis temas de negocio mediante patrones bilingües: comida,
servicio, ambiente, precio, espera y limpieza. Para cada restaurante se
cuentan las reseñas positivas, neutrales y negativas que mencionan cada
tema. La aplicación solo interpreta temas históricos con al menos cinco
menciones. Para estudiar el deterioro, compara los temas entre las dos
ventanas de seis meses y exige tres menciones por ventana.

La polaridad corresponde al sentimiento global de la reseña; por tanto,
los resultados representan señales para revisión y no relaciones causales.
Las recomendaciones se generan mediante reglas transparentes asociadas a
cada tema y se presentan como sugerencias operativas.


In [52]:
# Definir los temas de negocio
aspect_patterns = {
    'Comida y sabor': (
        r'\b(?:comida|food|platos?|dishes?|sabor(?:es)?|taste|'
        r'cocina|cuisine|ingredientes?|delicios[oa]s?|tasty)\b'
    ),
    'Servicio y atención': (
        r'\b(?:servicio|service|atenci[oó]n|camarer[oa]s?|waiter|'
        r'waitress|staff|personal|trato)\b'
    ),
    'Ambiente': (
        r'\b(?:ambiente|atmosphere|ambience|decoraci[oó]n|d[eé]cor|'
        r'terraza|terrace|m[uú]sica|music)\b'
    ),
    'Precio y valor': (
        r'\b(?:precio|prices?|car[oa]s?|expensive|barat[oa]s?|cheap|'
        r'calidad[ -]precio|value(?: for money)?)\b'
    ),
    'Espera y rapidez': (
        r'\b(?:espera|esperar|tard[óo]|demora|lento|slow|r[aá]pid[oa]|'
        r'fast|waiting?)\b'
    ),
    'Limpieza': (
        r'\b(?:limpieza|limpi[oa]s?|suci[oa]s?|higiene|clean|dirty)\b'
    )
}

# Preparar el texto y las fechas
aspect_source = modeling_corpus[
    ['review_id', 'item_id', 'date', 'review_text', 'sentiment']
].copy()
aspect_source['item_id'] = aspect_source['item_id'].astype(str)
aspect_source['date'] = pd.to_datetime(aspect_source['date'])
aspect_source['review_text'] = (
    aspect_source['review_text']
    .astype('string')
    .str.lower()
)

# Reutilizar las ventanas temporales del Notebook 2
last_observed_month = aspect_source['date'].max().to_period('M')
last_complete_month = last_observed_month - 1
recent_start = (last_complete_month - 5).to_timestamp()
recent_end = (last_complete_month + 1).to_timestamp()
previous_start = (last_complete_month - 11).to_timestamp()
previous_end = recent_start


def summarize_aspect_window(data, start, end, prefix):
    """Resumir un tema dentro de una ventana."""

    window = data.loc[
        data['date'].ge(start)
        & data['date'].lt(end)
    ]
    summary = (
        window
        .groupby('item_id', as_index=False)
        .agg(
            **{
                f'{prefix}_mentions': ('review_id', 'nunique'),
                f'{prefix}_negative_reviews': (
                    'negative_reviews',
                    'sum'
                )
            }
        )
    )
    summary[f'{prefix}_negative_percentage'] = (
        summary[f'{prefix}_negative_reviews']
        .div(summary[f'{prefix}_mentions'])
        .mul(100)
    )
    return summary


# Resumir cada tema por restaurante
aspect_summaries = []

for aspect, pattern in aspect_patterns.items():
    aspect_mask = aspect_source['review_text'].str.contains(
        pattern,
        regex=True,
        na=False
    )
    aspect_reviews = aspect_source.loc[
        aspect_mask,
        ['review_id', 'item_id', 'date', 'sentiment']
    ].copy()

    aspect_reviews['negative_reviews'] = (
        aspect_reviews['sentiment'].eq('Negativa').astype('int8')
    )
    aspect_reviews['neutral_reviews'] = (
        aspect_reviews['sentiment'].eq('Neutral').astype('int8')
    )
    aspect_reviews['positive_reviews'] = (
        aspect_reviews['sentiment'].eq('Positiva').astype('int8')
    )

    aspect_summary = (
        aspect_reviews
        .groupby('item_id', as_index=False)
        .agg(
            mentions=('review_id', 'nunique'),
            negative_reviews=('negative_reviews', 'sum'),
            neutral_reviews=('neutral_reviews', 'sum'),
            positive_reviews=('positive_reviews', 'sum')
        )
    )
    previous_summary = summarize_aspect_window(
        aspect_reviews,
        previous_start,
        previous_end,
        'previous'
    )
    recent_summary = summarize_aspect_window(
        aspect_reviews,
        recent_start,
        recent_end,
        'recent'
    )
    aspect_summary = (
        aspect_summary
        .merge(previous_summary, on='item_id', how='left')
        .merge(recent_summary, on='item_id', how='left')
    )
    aspect_summary['aspect'] = aspect
    aspect_summaries.append(aspect_summary)

restaurant_review_aspects = pd.concat(
    aspect_summaries,
    ignore_index=True
)

# Calcular porcentajes históricos
for sentiment in ['negative', 'neutral', 'positive']:
    restaurant_review_aspects[f'{sentiment}_percentage'] = (
        restaurant_review_aspects[f'{sentiment}_reviews']
        .div(restaurant_review_aspects['mentions'])
        .mul(100)
    )

# Calcular el cambio temático reciente
window_count_columns = [
    'previous_mentions',
    'previous_negative_reviews',
    'recent_mentions',
    'recent_negative_reviews'
]
restaurant_review_aspects[window_count_columns] = (
    restaurant_review_aspects[window_count_columns]
    .fillna(0)
    .astype('int32')
)
restaurant_review_aspects['minimum_window_mentions'] = (
    restaurant_review_aspects[
        ['previous_mentions', 'recent_mentions']
    ].min(axis=1)
)
restaurant_review_aspects['temporal_aspect_evaluable'] = (
    restaurant_review_aspects['minimum_window_mentions'].ge(3)
)
restaurant_review_aspects['temporal_negative_change'] = (
    restaurant_review_aspects['recent_negative_percentage']
    - restaurant_review_aspects['previous_negative_percentage']
)

# Clasificar el soporte histórico
restaurant_review_aspects['support_level'] = np.select(
    [
        restaurant_review_aspects['mentions'].ge(30),
        restaurant_review_aspects['mentions'].ge(10),
        restaurant_review_aspects['mentions'].ge(5)
    ],
    ['Alto', 'Medio', 'Bajo'],
    default='Insuficiente'
)

restaurant_review_aspects = restaurant_review_aspects[
    [
        'item_id',
        'aspect',
        'mentions',
        'negative_reviews',
        'negative_percentage',
        'neutral_reviews',
        'neutral_percentage',
        'positive_reviews',
        'positive_percentage',
        'support_level',
        'previous_mentions',
        'previous_negative_reviews',
        'previous_negative_percentage',
        'recent_mentions',
        'recent_negative_reviews',
        'recent_negative_percentage',
        'minimum_window_mentions',
        'temporal_aspect_evaluable',
        'temporal_negative_change'
    ]
].sort_values(['item_id', 'mentions'], ascending=[True, False])

# Guardar el producto analítico
review_aspects_path = DATA_PATH / 'restaurant_review_aspects.parquet'
restaurant_review_aspects.to_parquet(
    review_aspects_path,
    index=False,
    engine='pyarrow'
)

aspect_validation = pd.Series(
    {
        'file': review_aspects_path.name,
        'exists': review_aspects_path.exists(),
        'rows': len(restaurant_review_aspects),
        'restaurants': restaurant_review_aspects['item_id'].nunique(),
        'aspects': restaurant_review_aspects['aspect'].nunique(),
        'supported_rows': (
            restaurant_review_aspects['mentions'].ge(5).sum()
        ),
        'temporal_supported_rows': (
            restaurant_review_aspects[
                'temporal_aspect_evaluable'
            ].sum()
        ),
        'previous_window_start': previous_start.date(),
        'recent_window_start': recent_start.date(),
        'recent_window_end': (
            recent_end - pd.Timedelta(days=1)
        ).date()
    },
    name='result'
)

display(aspect_validation.to_frame())


,result
file,restaurant_review_aspects.parquet
exists,True
rows,47649
restaurants,9919
aspects,6
supported_rows,29556
temporal_supported_rows,3153
previous_window_start,2022-06-01
recent_window_start,2022-12-01
recent_window_end,2023-05-31


### 7.7.2. Construcción de la interfaz

Streamlit reutiliza los módulos de inferencia, explicabilidad, benchmarking
e interpretación temática. La barra lateral permite alternar entre el
diagnóstico individual y un explorador competitivo independiente. La vista
individual explica el diagnóstico, los pares, las alertas, las señales de
deterioro y el origen metodológico de cada resultado.


In [53]:
%%writefile app/ui.py

import streamlit as st

app_css = """
<style>
:root {
    --forest: #05402B;
    --forest_soft: #1D5A43;
    --sage: #B8C8BB;
    --ivory: #F7F5EF;
    --paper: #FFFEFA;
    --ink: #183027;
    --muted: #6F7B74;
    --gold: #A8874F;
}

html,
body,
[class*='css'] {
    font-family:
        -apple-system,
        BlinkMacSystemFont,
        'Helvetica Neue',
        Arial,
        sans-serif;
    color: var(--ink);
}

.stApp {
    background:
        radial-gradient(
            circle at 92% 0%,
            rgba(184, 200, 187, 0.34),
            transparent 28rem
        ),
        var(--ivory);
}

[data-testid='stHeader'] {
    background: rgba(247, 245, 239, 0.88);
    border-bottom: 1px solid rgba(5, 64, 43, 0.08);
    backdrop-filter: blur(18px);
}

.block-container {
    max-width: 1240px;
    padding-top: 2.8rem;
    padding-bottom: 4rem;
}

h1,
h2,
h3 {
    color: var(--forest);
    font-family:
        Georgia,
        'Times New Roman',
        serif;
    letter-spacing: -0.025em;
}

.hero {
    padding: 3.6rem 3.8rem;
    margin-bottom: 1.5rem;
    border: 1px solid rgba(5, 64, 43, 0.10);
    border-radius: 28px;
    background:
        linear-gradient(
            125deg,
            rgba(255, 254, 250, 0.98),
            rgba(229, 235, 227, 0.88)
        );
    box-shadow:
        0 24px 70px rgba(5, 64, 43, 0.08);
}

.hero-label {
    margin-bottom: 1rem;
    color: var(--gold);
    font-size: 0.74rem;
    font-weight: 700;
    letter-spacing: 0.18em;
    text-transform: uppercase;
}

.hero h1 {
    max-width: 780px;
    margin: 0;
    font-size: clamp(
        2.6rem,
        5vw,
        4.8rem
    );
    font-weight: 500;
    line-height: 0.98;
}

.hero p {
    max-width: 720px;
    margin: 1.4rem 0 0;
    color: var(--muted);
    font-size: 1.06rem;
    line-height: 1.7;
}

[data-testid='stMetric'] {
    min-height: 126px;
    padding: 1.25rem 1.35rem;
    border: 1px solid rgba(5, 64, 43, 0.10);
    border-radius: 20px;
    background: rgba(255, 254, 250, 0.88);
    box-shadow:
        0 12px 30px rgba(5, 64, 43, 0.05);
}

[data-testid='stMetricLabel'] {
    color: var(--muted);
    font-size: 0.76rem;
    font-weight: 650;
    letter-spacing: 0.06em;
    text-transform: uppercase;
}

[data-testid='stMetricValue'] {
    color: var(--forest);
    font-family:
        Georgia,
        'Times New Roman',
        serif;
    font-size: 2rem;
}

[data-testid='stSidebar'] {
    background: var(--forest);
    border-right: 0;
}

[data-testid='stSidebar'] h1,
[data-testid='stSidebar'] h2,
[data-testid='stSidebar'] h3,
[data-testid='stSidebar'] p,
[data-testid='stSidebar'] label {
    color: var(--ivory) !important;
}

[data-testid='stSidebar']
[data-baseweb='select'] * {
    color: var(--ink) !important;
}

.sidebar-brand {
    padding: 1.2rem 0 1.8rem;
    margin-bottom: 1.5rem;
    border-bottom:
        1px solid rgba(255, 255, 255, 0.18);
}

.sidebar-brand strong {
    display: block;
    color: var(--ivory);
    font-family:
        Georgia,
        'Times New Roman',
        serif;
    font-size: 1.7rem;
    font-weight: 500;
}

.sidebar-brand span {
    display: block;
    margin-top: 0.45rem;
    color: rgba(247, 245, 239, 0.66);
    font-size: 0.72rem;
    letter-spacing: 0.13em;
    text-transform: uppercase;
}

.stTabs [data-baseweb='tab-list'] {
    gap: 0.4rem;
    padding: 0.35rem;
    border: 1px solid rgba(5, 64, 43, 0.09);
    border-radius: 16px;
    background: rgba(255, 254, 250, 0.72);
}

.stTabs [data-baseweb='tab'] {
    height: 44px;
    padding: 0 1rem;
    border-radius: 12px;
    color: var(--muted);
    font-weight: 600;
}

.stTabs [aria-selected='true'] {
    color: var(--ivory) !important;
    background: var(--forest);
}

.stTabs [data-baseweb='tab-highlight'] {
    display: none;
}

.stButton > button {
    min-height: 46px;
    border: 1px solid var(--forest);
    border-radius: 999px;
    color: var(--ivory);
    background: var(--forest);
    font-weight: 650;
    transition:
        transform 160ms ease,
        box-shadow 160ms ease;
}

.stButton > button:hover {
    border-color: var(--forest_soft);
    color: var(--ivory);
    background: var(--forest_soft);
    box-shadow:
        0 10px 24px rgba(5, 64, 43, 0.16);
    transform: translateY(-1px);
}

[data-baseweb='select'] > div,
[data-baseweb='input'] > div,
textarea {
    border-color:
        rgba(5, 64, 43, 0.14) !important;
    border-radius: 14px !important;
    background: var(--paper) !important;
}

[data-testid='stDataFrame'] {
    overflow: hidden;
    border:
        1px solid rgba(5, 64, 43, 0.10);
    border-radius: 18px;
    background: var(--paper);
}

[data-testid='stAlert'] {
    border-radius: 16px;
}

@media (max-width: 760px) {
    .block-container {
        padding-top: 1.4rem;
    }

    .hero {
        padding: 2.2rem 1.5rem;
        border-radius: 22px;
    }
}


/* Plan de acción */
.action-section,
.strength-section,
.driver-section {
    margin: 2.4rem 0 1rem;
}

.action-section h2,
.strength-section h2,
.driver-section h2 {
    color: #183027;
    font-family: Georgia, 'Times New Roman', serif;
    font-size: 2rem;
    margin: 0.2rem 0 0.35rem;
}

.action-section p,
.driver-section p {
    color: #66736D;
    margin: 0;
    max-width: 760px;
}

.section-eyebrow,
.card-label {
    color: #A8874F;
    font-size: 0.70rem;
    font-weight: 750;
    letter-spacing: 0.14em;
    text-transform: uppercase;
}

.priority-card {
    background: linear-gradient(135deg, #FFFEFB 0%, #FBF4EF 100%);
    border: 1px solid rgba(139, 30, 30, 0.14);
    border-left: 5px solid #8B1E1E;
    border-radius: 20px;
    box-shadow: 0 14px 32px rgba(24, 48, 39, 0.07);
    margin: 1rem 0;
    padding: 1.45rem;
}

.priority-card__head {
    align-items: flex-start;
    display: flex;
    gap: 1rem;
    margin-bottom: 1rem;
}

.priority-card h3,
.strength-card h3 {
    color: #183027;
    font-family: Georgia, 'Times New Roman', serif;
    font-size: 1.35rem;
    line-height: 1.25;
    margin: 0.15rem 0 0;
}

.priority-number {
    align-items: center;
    background: #8B1E1E;
    border-radius: 999px;
    color: #FFFFFF;
    display: inline-flex;
    flex: 0 0 2.65rem;
    font-size: 0.82rem;
    font-weight: 750;
    height: 2.65rem;
    justify-content: center;
    letter-spacing: 0.06em;
}

.priority-evidence {
    background: rgba(139, 30, 30, 0.055);
    border-radius: 14px;
    margin-bottom: 0.85rem;
    padding: 0.9rem 1rem;
}

.priority-evidence p,
.priority-grid p,
.strength-card p {
    color: #3E4B46;
    line-height: 1.55;
    margin: 0.3rem 0 0;
}

.priority-grid {
    display: grid;
    gap: 0.85rem;
    grid-template-columns: repeat(2, minmax(0, 1fr));
}

.priority-grid > div {
    background: rgba(255, 255, 255, 0.82);
    border: 1px solid rgba(5, 64, 43, 0.10);
    border-radius: 14px;
    padding: 0.9rem 1rem;
}

.strength-card {
    background: linear-gradient(135deg, #F9FCF8 0%, #EEF5EF 100%);
    border: 1px solid rgba(5, 64, 43, 0.13);
    border-left: 5px solid #05402B;
    border-radius: 18px;
    margin: 0.75rem 0;
    padding: 1.1rem 1.25rem;
}

/* Señales de deterioro */
.driver-card {
    align-items: center;
    background: #FFFFFF;
    border: 1px solid rgba(139, 30, 30, 0.14);
    border-radius: 18px;
    display: grid;
    gap: 1.2rem;
    grid-template-columns: 1fr 1.3fr;
    margin: 0.8rem 0;
    padding: 1.15rem 1.25rem;
}

.driver-card h3 {
    color: #183027;
    font-family: Georgia, 'Times New Roman', serif;
    margin: 0.2rem 0 0;
}

.driver-card p {
    color: #56635E;
    margin: 0.45rem 0 0;
}

.driver-comparison {
    align-items: center;
    background: #FBF7F3;
    border-radius: 14px;
    display: grid;
    gap: 0.7rem;
    grid-template-columns: 1fr auto 1fr;
    padding: 0.85rem;
    text-align: center;
}

.driver-comparison div:not(.driver-arrow) {
    display: flex;
    flex-direction: column;
}

.driver-comparison span,
.driver-comparison small {
    color: #69746F;
    font-size: 0.66rem;
}

.driver-comparison strong {
    color: #8B1E1E;
    font-size: 1.25rem;
    margin: 0.2rem 0;
}

.driver-arrow {
    color: #A8874F;
    font-size: 1.4rem;
}

.decision-summary {
    background: linear-gradient(135deg, #05402B 0%, #1D5A43 100%);
    border-radius: 20px;
    box-shadow: 0 16px 34px rgba(5, 64, 43, 0.16);
    margin: 1.2rem 0;
    padding: 1.35rem 1.5rem;
}

.decision-summary .card-label {
    color: #D4BA88;
}

.decision-summary p {
    color: #FFFFFF;
    font-family: Georgia, 'Times New Roman', serif;
    font-size: 1.15rem;
    line-height: 1.6;
    margin: 0.45rem 0 0;
}

/* Explorador de mercado */
.market-hero {
    background: linear-gradient(135deg, #FFFEFB 0%, #EAF2EC 100%);
    border: 1px solid rgba(5, 64, 43, 0.10);
    border-radius: 24px;
    margin: 1rem 0 1.3rem;
    padding: 2rem;
}

.market-hero h1 {
    color: #183027;
    font-family: Georgia, 'Times New Roman', serif;
    font-size: 2.4rem;
    line-height: 1.1;
    margin: 0.35rem 0 0.75rem;
}

.market-hero p {
    color: #5E6D66;
    line-height: 1.6;
    margin: 0;
    max-width: 760px;
}

section[data-testid="stSidebar"] div.stButton > button {
    border-radius: 12px;
    font-weight: 650;
    min-height: 2.8rem;
}

@media (max-width: 760px) {
    .priority-grid,
    .driver-card {
        grid-template-columns: 1fr;
    }
}

</style>
"""


def apply_style():
    """Aplicar el diseño visual."""

    st.markdown(
        app_css,
        unsafe_allow_html=True
    )
def render_header():
    """Mostrar la cabecera."""

    hero_html = (
        '<section class="hero">'
        '<div class="hero-label">'
        'Madrid · Restaurant Intelligence'
        '</div>'
        '<h1>'
        'Reputación que se convierte en decisiones.'
        '</h1>'
        '<p>'
        'Diagnóstico, comparación competitiva y alertas '
        'tempranas construidas a partir de la voz real '
        'de los clientes.'
        '</p>'
        '</section>'
    )

    sidebar_html = (
        '<div class="sidebar-brand">'
        '<strong>'
        'Madrid Intelligence'
        '</strong>'
        '<span>'
        'Restaurant reputation system'
        '</span>'
        '</div>'
    )

    st.markdown(
        hero_html,
        unsafe_allow_html=True
    )

    st.sidebar.markdown(
        sidebar_html,
        unsafe_allow_html=True
    )

    st.sidebar.caption(
        'Selecciona un restaurante para comenzar.'
    )
def style_figure(figure):
    """Aplicar el estilo a Plotly."""

    figure.update_layout(
        paper_bgcolor='rgba(0, 0, 0, 0)',
        plot_bgcolor=(
            'rgba(255, 254, 250, 0.72)'
        ),
        font={
            'family': (
                '-apple-system, '
                'BlinkMacSystemFont, '
                'Helvetica Neue'
            ),
            'color': '#183027'
        },
        margin={
            'l': 20,
            'r': 20,
            't': 35,
            'b': 20
        }
    )

    figure.update_xaxes(
        gridcolor='rgba(5, 64, 43, 0.08)'
    )

    figure.update_yaxes(
        gridcolor='rgba(5, 64, 43, 0.08)'
    )

    return figure

Overwriting app/ui.py


In [54]:
%%writefile app/benchmarking.py

import pandas as pd
import plotly.express as px
import streamlit as st

from app.ui import style_figure


primary_color = '#05402B'

top_ten_rules = {
    'Mejor rating': ('mean_review_rating', False, 'Rating'),
    'Mayor porcentaje positivo': (
        'positive_percentage',
        False,
        'Positivas'
    ),
    'Mayor volumen de reseñas': ('reviews', False, 'Reseñas'),
    'Mayor mejora reciente': ('rating_change', False, 'Cambio'),
    'Mayor deterioro reciente': ('rating_change', True, 'Cambio')
}

criterion_explanations = {
    'Mejor rating': (
        'Ordena por la valoración media de las reseñas. Un valor mayor indica '
        'una mejor experiencia general reportada por los clientes.'
    ),
    'Mayor porcentaje positivo': (
        'Ordena por la proporción de reseñas clasificadas como positivas. '
        'Permite comparar consistencia, no solo el promedio de estrellas.'
    ),
    'Mayor volumen de reseñas': (
        'Ordena por cantidad de reseñas. Refleja visibilidad y actividad, '
        'pero un volumen alto no implica mejor calidad.'
    ),
    'Mayor mejora reciente': (
        'Ordena por el aumento del rating entre dos ventanas consecutivas de '
        'seis meses completos. Solo incluye restaurantes evaluables.'
    ),
    'Mayor deterioro reciente': (
        'Ordena por la mayor caída del rating entre las dos ventanas. Es una '
        'señal de cambio observado, no una predicción.'
    )
}


def get_restaurant_groups(restaurant, membership, groups):
    """Obtener los grupos del restaurante."""

    restaurant_groups = (
        membership.loc[
            membership['item_id'].eq(restaurant['item_id'])
        ]
        .merge(
            groups,
            on=['specialty', 'price_interval'],
            how='left',
            validate='many_to_one'
        )
    )

    if restaurant_groups.empty:
        return restaurant_groups

    restaurant_groups['group_label'] = (
        restaurant_groups['specialty']
        + ' | '
        + restaurant_groups['price_interval']
    )
    restaurant_groups['rating_gap'] = (
        restaurant['mean_review_rating']
        - restaurant_groups['group_mean_rating']
    )
    restaurant_groups['negative_gap'] = (
        restaurant['negative_percentage']
        - restaurant_groups['group_negative_percentage']
    )
    return restaurant_groups


def get_group_restaurants(
    specialty,
    price_interval,
    membership,
    diagnostics,
    minimum_reviews=30
):
    """Obtener restaurantes comparables."""

    group_members = membership.loc[
        membership['specialty'].eq(specialty)
        & membership['price_interval'].eq(price_interval),
        ['item_id']
    ].drop_duplicates()

    return (
        group_members
        .merge(
            diagnostics,
            on='item_id',
            how='left',
            validate='one_to_one'
        )
        .loc[lambda data: data['reviews'].ge(minimum_reviews)]
        .copy()
    )


def build_executive_summary(restaurant, restaurant_groups):
    """Explicar el diagnóstico principal."""

    support_text = (
        f"El diagnóstico se basa en {int(restaurant['reviews']):,} reseñas "
        f"y tiene soporte {restaurant['diagnostic_support']}."
    )

    if restaurant_groups.empty:
        comparison_text = (
            ' No existe un grupo competitivo elegible para este restaurante.'
        )
    else:
        group = (
            restaurant_groups
            .sort_values('group_restaurants', ascending=False)
            .iloc[0]
        )
        rating_gap = float(group['rating_gap'])
        negative_gap = float(group['negative_gap'])
        rating_text = (
            f"supera a sus pares por {abs(rating_gap):.2f} puntos"
            if rating_gap >= 0
            else f"está por debajo de sus pares por {abs(rating_gap):.2f} puntos"
        )
        negative_text = (
            f"tiene {abs(negative_gap):.2f} puntos porcentuales menos de negativas"
            if negative_gap <= 0
            else f"tiene {abs(negative_gap):.2f} puntos porcentuales más de negativas"
        )
        comparison_text = (
            f" Frente al grupo {group['group_label']}, {rating_text} y "
            f"{negative_text}."
        )

    if not bool(restaurant['alert_evaluable']):
        alert_text = (
            ' La evolución reciente todavía no puede evaluarse con soporte suficiente.'
        )
    elif restaurant['alert_level'] == 'Sin alerta':
        alert_text = ' No se detectan señales recientes de deterioro.'
    else:
        alert_text = (
            f" La señal actual es {restaurant['alert_level']} y la prioridad "
            f"es {restaurant['operational_priority']}."
        )

    return support_text + comparison_text + alert_text


def render_benchmarking(
    restaurant,
    restaurant_groups,
    membership,
    diagnostics
):
    """Mostrar la comparación con pares."""

    st.header('Comparación con grupos similares')

    with st.expander('Qué significa esta comparación'):
        st.markdown(
            '- **Pares:** restaurantes de la misma especialidad y rango de '
            'precio.\n'
            '- **Posición:** lugar por rating entre pares con al menos 30 '
            'reseñas.\n'
            '- **Brecha de rating:** positiva significa mejor desempeño.\n'
            '- **Brecha de negativas:** positiva significa peor desempeño.'
        )

    if restaurant_groups.empty:
        st.info('El restaurante no pertenece a un grupo competitivo elegible.')
        return

    selected_label = st.selectbox(
        'Grupo de comparación',
        restaurant_groups['group_label'].tolist(),
        key='benchmark_group'
    )
    group = restaurant_groups.loc[
        restaurant_groups['group_label'].eq(selected_label)
    ].iloc[0]

    peers = get_group_restaurants(
        group['specialty'],
        group['price_interval'],
        membership,
        diagnostics
    )
    ranked_peers = (
        peers
        .dropna(subset=['mean_review_rating'])
        .sort_values(
            ['mean_review_rating', 'reviews'],
            ascending=[False, False]
        )
        .reset_index(drop=True)
    )
    rank_match = ranked_peers.index[
        ranked_peers['item_id'].eq(restaurant['item_id'])
    ].tolist()
    position = (
        f'{rank_match[0] + 1} de {len(ranked_peers)}'
        if rank_match
        else 'Sin soporte'
    )

    metrics = st.columns(4)
    metrics[0].metric('Posición', position)
    metrics[1].metric('Pares con soporte', f'{len(ranked_peers):,}')
    metrics[2].metric('Brecha de rating', f"{group['rating_gap']:.2f}")
    metrics[3].metric(
        'Brecha de negativas',
        f"{group['negative_gap']:.2f} pp"
    )

    rating_data = pd.DataFrame(
        {
            'Referencia': ['Restaurante', 'Pares'],
            'Valor': [
                restaurant['mean_review_rating'],
                group['group_mean_rating']
            ]
        }
    )
    negative_data = pd.DataFrame(
        {
            'Referencia': ['Restaurante', 'Pares'],
            'Valor': [
                restaurant['negative_percentage'],
                group['group_negative_percentage']
            ]
        }
    )

    rating_column, negative_column = st.columns(2)
    rating_figure = px.bar(
        rating_data,
        x='Referencia',
        y='Valor',
        color='Referencia',
        text='Valor',
        title='Rating medio: mayor es mejor',
        color_discrete_map={
            'Restaurante': primary_color,
            'Pares': '#B8C8BB'
        }
    )
    rating_figure.update_traces(
        texttemplate='%{text:.2f}',
        textposition='outside'
    )
    rating_figure.update_layout(showlegend=False)
    rating_figure.update_yaxes(range=[0, 5.4])
    rating_column.plotly_chart(
        style_figure(rating_figure),
        use_container_width=True
    )

    negative_figure = px.bar(
        negative_data,
        x='Referencia',
        y='Valor',
        color='Referencia',
        text='Valor',
        title='Reseñas negativas: menor es mejor',
        color_discrete_map={
            'Restaurante': '#8B1E1E',
            'Pares': '#D8D3C8'
        }
    )
    negative_figure.update_traces(
        texttemplate='%{text:.2f}%',
        textposition='outside'
    )
    negative_figure.update_layout(showlegend=False)
    negative_figure.update_yaxes(
        range=[0, min(100, max(20, negative_data['Valor'].max() * 1.25))]
    )
    negative_column.plotly_chart(
        style_figure(negative_figure),
        use_container_width=True
    )

    if group['rating_gap'] < 0 and group['negative_gap'] > 0:
        message = (
            'Prioridad de mejora: elevar el rating y reducir las reseñas '
            'negativas para cerrar la brecha con los pares.'
        )
    elif group['rating_gap'] < 0:
        message = (
            'El porcentaje negativo es competitivo, pero el rating permanece '
            'por debajo del grupo.'
        )
    elif group['negative_gap'] > 0:
        message = (
            'El rating es competitivo, pero las reseñas negativas superan '
            'la referencia del grupo.'
        )
    else:
        message = (
            'El restaurante supera a sus pares en rating y en proporción '
            'de reseñas negativas.'
        )
    st.info(message)

    with st.expander('Ver todos los grupos del restaurante'):
        table = restaurant_groups[
            [
                'group_label',
                'group_restaurants',
                'group_reviews',
                'rating_gap',
                'negative_gap'
            ]
        ].rename(
            columns={
                'group_label': 'Grupo competitivo',
                'group_restaurants': 'Restaurantes',
                'group_reviews': 'Reseñas',
                'rating_gap': 'Brecha de rating',
                'negative_gap': 'Brecha de negativas'
            }
        )
        st.dataframe(
            table.round(2),
            hide_index=True,
            use_container_width=True
        )


def get_top_ranking(
    specialty,
    price_interval,
    criterion,
    membership,
    diagnostics
):
    """Construir el ranking del grupo."""

    column, ascending, label = top_ten_rules[criterion]
    ranking = get_group_restaurants(
        specialty,
        price_interval,
        membership,
        diagnostics
    ).dropna(subset=[column])
    if column == 'rating_change':
        ranking = ranking.loc[ranking['alert_evaluable']]
    ranking = (
        ranking
        .sort_values(
            [column, 'reviews'],
            ascending=[ascending, False]
        )
        .reset_index(drop=True)
    )
    ranking['position'] = ranking.index + 1
    return ranking, column, label


def render_market_explorer(
    groups,
    membership,
    diagnostics
):
    """Mostrar el explorador competitivo."""

    specialties = sorted(groups['specialty'].dropna().unique())

    st.markdown(
        '''
        <section class="market-hero">
            <span class="section-eyebrow">EXPLORADOR DE MERCADO</span>
            <h1>Conoce la competencia de Madrid</h1>
            <p>
                Compara restaurantes de la misma especialidad y rango de
                precio. Cambia los filtros para descubrir referentes,
                volumen de actividad y señales recientes del mercado.
            </p>
        </section>
        ''',
        unsafe_allow_html=True
    )

    guide_columns = st.columns(3)
    guide_columns[0].info(
        '**Quiénes son comparables**\n\n'
        'Restaurantes que comparten especialidad y rango de precio.'
    )
    guide_columns[1].info(
        '**Qué entra al ranking**\n\n'
        'Solo restaurantes con al menos 30 reseñas.'
    )
    guide_columns[2].info(
        '**Cómo debe usarse**\n\n'
        'Como referencia competitiva, no como recomendación comercial.'
    )

    filter_columns = st.columns(3)
    specialty = filter_columns[0].selectbox(
        'Especialidad',
        specialties,
        key='market_specialty'
    )
    price_options = sorted(
        groups.loc[
            groups['specialty'].eq(specialty),
            'price_interval'
        ].dropna().unique()
    )
    price_interval = filter_columns[1].selectbox(
        'Rango de precio',
        price_options,
        key='market_price'
    )
    criterion = filter_columns[2].selectbox(
        'Criterio del Top 10',
        list(top_ten_rules),
        key='market_criterion'
    )

    st.info(criterion_explanations[criterion])

    ranking, column, label = get_top_ranking(
        specialty,
        price_interval,
        criterion,
        membership,
        diagnostics
    )

    if ranking.empty:
        st.warning('No existen restaurantes con soporte para este criterio.')
        return

    selected_group = groups.loc[
        groups['specialty'].eq(specialty)
        & groups['price_interval'].eq(price_interval)
    ].iloc[0]

    market_metrics = st.columns(4)
    market_metrics[0].metric(
        'Restaurantes comparables',
        f'{len(ranking):,}'
    )
    market_metrics[1].metric(
        'Reseñas del grupo',
        f"{int(selected_group['group_reviews']):,}"
    )
    market_metrics[2].metric(
        'Rating medio del grupo',
        f"{selected_group['group_mean_rating']:.2f}"
    )
    market_metrics[3].metric(
        'Negativas del grupo',
        f"{selected_group['group_negative_percentage']:.2f} %"
    )
    st.caption(
        'Los dos promedios del grupo se calculan a nivel de restaurante para '
        'evitar que los establecimientos con más reseñas dominen la comparación.'
    )

    top_ten = ranking.head(10).copy()
    top_ten['restaurant_name'] = top_ten['name'].astype(str).str.slice(0, 42)
    top_ten['ranking_value'] = top_ten[column]
    chart_data = top_ten.sort_values('position', ascending=False)

    ranking_figure = px.bar(
        chart_data,
        x='ranking_value',
        y='restaurant_name',
        orientation='h',
        text='ranking_value',
        color_discrete_sequence=[primary_color],
        labels={
            'ranking_value': label,
            'restaurant_name': 'Restaurante'
        },
        title=f'Top 10: {criterion.lower()}'
    )
    if column == 'reviews':
        ranking_figure.update_traces(texttemplate='%{text:,.0f}')
    elif column == 'positive_percentage':
        ranking_figure.update_traces(texttemplate='%{text:.1f}%')
    else:
        ranking_figure.update_traces(texttemplate='%{text:.2f}')
    ranking_figure.update_traces(textposition='outside')
    st.plotly_chart(
        style_figure(ranking_figure),
        use_container_width=True
    )

    table_columns = [
        'position',
        'name',
        'reviews',
        'mean_review_rating',
        'positive_percentage',
        'negative_percentage'
    ]
    if column not in table_columns:
        table_columns.insert(2, column)
    ranking_table = top_ten[table_columns].rename(
        columns={
            'position': 'Posición',
            'name': 'Restaurante',
            'reviews': 'Reseñas',
            'mean_review_rating': 'Rating',
            'positive_percentage': 'Positivas (%)',
            'negative_percentage': 'Negativas (%)',
            'rating_change': 'Cambio de rating'
        }
    )
    st.dataframe(
        ranking_table.round(2),
        hide_index=True,
        use_container_width=True
    )

    with st.expander('De dónde salen estos resultados'):
        st.markdown(
            '- Los restaurantes proceden del conjunto depurado de Madrid.\n'
            '- Un grupo combina **especialidad** y **rango de precio**.\n'
            '- Los grupos elegibles contienen al menos 30 restaurantes y '
            '1,000 reseñas en conjunto.\n'
            '- El Top 10 exige al menos 30 reseñas por restaurante.\n'
            '- Los criterios recientes usan dos ventanas consecutivas de '
            'seis meses completos.\n'
            '- El ranking describe los datos observados y no garantiza '
            'calidad futura, ventas ni rentabilidad.'
        )


Overwriting app/benchmarking.py


In [55]:
%%writefile app/insights.py

from html import escape

import pandas as pd
import streamlit as st


aspect_actions = {
    'Comida y sabor': {
        'action': (
            'Revisar los platos citados en las reseñas negativas y comprobar '
            'la consistencia del sabor, la temperatura y la presentación.'
        ),
        'measurement': (
            'Comprobar si disminuye el porcentaje de reseñas negativas que '
            'mencionan comida y sabor.'
        )
    },
    'Servicio y atención': {
        'action': (
            'Revisar la atención en horas de mayor demanda, la asignación de '
            'personal y los protocolos de seguimiento al cliente.'
        ),
        'measurement': (
            'Comprobar si disminuyen las menciones negativas sobre servicio '
            'en las reseñas posteriores.'
        )
    },
    'Ambiente': {
        'action': (
            'Revisar los comentarios sobre ruido, iluminación, comodidad, '
            'música, terraza y distribución del espacio.'
        ),
        'measurement': (
            'Comprobar si aumenta la proporción de reseñas positivas que '
            'mencionan el ambiente.'
        )
    },
    'Precio y valor': {
        'action': (
            'Revisar la relación percibida entre precio, porciones, calidad '
            'y experiencia, además de la claridad de la carta.'
        ),
        'measurement': (
            'Comprobar si disminuyen las menciones negativas relacionadas '
            'con precio y valor.'
        )
    },
    'Espera y rapidez': {
        'action': (
            'Medir los tiempos de espera y revisar la coordinación entre '
            'reservas, cocina y sala durante los periodos de mayor demanda.'
        ),
        'measurement': (
            'Comprobar si disminuyen las reseñas negativas que mencionan '
            'esperas, demoras o lentitud.'
        )
    },
    'Limpieza': {
        'action': (
            'Aplicar listas de comprobación y controles periódicos en sala, '
            'baños y zonas visibles para el cliente.'
        ),
        'measurement': (
            'Comprobar si dejan de aparecer menciones negativas sobre '
            'limpieza e higiene.'
        )
    }
}


def get_supported_aspects(
    review_aspects,
    item_id,
    minimum_mentions=5
):
    """Obtener temas con soporte mínimo."""

    return review_aspects.loc[
        review_aspects['item_id'].eq(str(item_id))
        & review_aspects['mentions'].ge(minimum_mentions)
    ].copy()


def get_temporal_drivers(aspects):
    """Obtener temas negativos que aumentaron recientemente."""

    required_columns = {
        'temporal_aspect_evaluable',
        'temporal_negative_change',
        'previous_mentions',
        'previous_negative_percentage',
        'recent_mentions',
        'recent_negative_reviews',
        'recent_negative_percentage'
    }
    if not required_columns.issubset(aspects.columns):
        return aspects.iloc[0:0].copy()

    return (
        aspects.loc[
            aspects['temporal_aspect_evaluable']
            & aspects['temporal_negative_change'].gt(0)
            & aspects['recent_negative_reviews'].gt(0)
        ]
        .sort_values(
            [
                'temporal_negative_change',
                'recent_negative_reviews',
                'recent_mentions'
            ],
            ascending=False
        )
        .copy()
    )


def get_reference_group(restaurant_groups):
    """Seleccionar el grupo con mayor soporte."""

    if restaurant_groups.empty:
        return None

    return (
        restaurant_groups
        .sort_values('group_restaurants', ascending=False)
        .iloc[0]
    )


def build_restaurant_insights(
    restaurant,
    restaurant_groups,
    review_aspects
):
    """Construir el plan de mejora y las fortalezas."""

    aspects = get_supported_aspects(
        review_aspects,
        restaurant['item_id']
    )
    priorities = []
    strengths = []
    reference_group = get_reference_group(restaurant_groups)
    temporal_drivers = get_temporal_drivers(aspects)
    temporal_driver_names = set(temporal_drivers['aspect'])

    if bool(restaurant['alert_evaluable']):
        alert_level = str(restaurant['alert_level'])
        if alert_level != 'Sin alerta':
            alert_scores = {
                'Vigilancia': 80,
                'Alerta alta': 90,
                'Alerta crítica': 100
            }
            minimum_reviews = int(
                restaurant['minimum_window_reviews']
            )
            priority_title = (
                'Atender el deterioro reciente'
                if minimum_reviews >= 10
                else 'Validar el deterioro reciente'
            )
            priorities.append(
                {
                    'title': priority_title,
                    'evidence': (
                        f"El rating cambió {restaurant['rating_change']:.2f} "
                        'puntos y las reseñas negativas cambiaron '
                        f"{restaurant['negative_percentage_point_change']:.2f} "
                        'puntos porcentuales. El soporte mínimo es de '
                        f'{minimum_reviews} reseñas por ventana.'
                    ),
                    'action': (
                        'Revisar las reseñas de la ventana reciente y '
                        'comprobar si el deterioro coincide con cambios en '
                        'personal, carta, horarios o funcionamiento.'
                    ),
                    'measurement': (
                        'Recalcular el indicador cuando existan nuevas '
                        'reseñas y comprobar si deja de cumplir los límites '
                        'de vigilancia.'
                    ),
                    'score': alert_scores.get(alert_level, 80)
                }
            )

            for row in temporal_drivers.head(2).itertuples():
                guidance = aspect_actions[row.aspect]
                priorities.append(
                    {
                        'title': f'Revisar {row.aspect.lower()}',
                        'evidence': (
                            'Las menciones negativas del tema aumentaron de '
                            f'{row.previous_negative_percentage:.1f} % en la '
                            'ventana anterior a '
                            f'{row.recent_negative_percentage:.1f} % en la '
                            'reciente '
                            f'({row.temporal_negative_change:+.1f} puntos). '
                            'Se observaron '
                            f'{int(row.previous_mentions)} y '
                            f'{int(row.recent_mentions)} menciones.'
                        ),
                        'action': guidance['action'],
                        'measurement': guidance['measurement'],
                        'score': 85 + row.temporal_negative_change
                    }
                )

    if not aspects.empty:
        aspects['negative_difference'] = (
            aspects['negative_percentage']
            - float(restaurant['negative_percentage'])
        )
        minimum_negative_percentage = max(
            20.00,
            float(restaurant['negative_percentage'])
        )
        negative_aspects = (
            aspects.loc[
                aspects['negative_reviews'].ge(3)
                & aspects['negative_percentage'].ge(
                    minimum_negative_percentage
                )
                & ~aspects['aspect'].isin(temporal_driver_names)
            ]
            .sort_values(
                [
                    'negative_difference',
                    'negative_reviews',
                    'mentions'
                ],
                ascending=False
            )
            .head(2)
        )

        for row in negative_aspects.itertuples():
            guidance = aspect_actions[row.aspect]
            priorities.append(
                {
                    'title': f'Mejorar {row.aspect.lower()}',
                    'evidence': (
                        f'{int(row.negative_reviews)} de las '
                        f'{int(row.mentions)} reseñas que mencionan este '
                        'tema son negativas '
                        f'({row.negative_percentage:.1f} %). El soporte es '
                        f'{row.support_level.lower()}.'
                    ),
                    'action': guidance['action'],
                    'measurement': guidance['measurement'],
                    'score': (
                        75
                        + max(0, row.negative_difference)
                        + min(row.mentions, 30) / 10
                    )
                }
            )

        aspects['positive_difference'] = (
            aspects['positive_percentage']
            - float(restaurant['positive_percentage'])
        )
        minimum_positive_percentage = max(
            50.00,
            float(restaurant['positive_percentage'])
        )
        positive_aspects = (
            aspects.loc[
                aspects['positive_reviews'].ge(3)
                & aspects['positive_percentage'].ge(
                    minimum_positive_percentage
                )
            ]
            .sort_values(
                [
                    'positive_difference',
                    'positive_reviews',
                    'mentions'
                ],
                ascending=False
            )
            .head(2)
        )

        for row in positive_aspects.itertuples():
            strengths.append(
                {
                    'title': row.aspect,
                    'evidence': (
                        f'{int(row.positive_reviews)} de las '
                        f'{int(row.mentions)} reseñas que mencionan este '
                        'tema son positivas '
                        f'({row.positive_percentage:.1f} %).'
                    )
                }
            )

    if reference_group is not None:
        if reference_group['rating_gap'] < -0.10:
            priorities.append(
                {
                    'title': 'Cerrar la brecha de rating',
                    'evidence': (
                        f"El rating está {abs(reference_group['rating_gap']):.2f} "
                        f"puntos por debajo del grupo "
                        f"{reference_group['group_label']}."
                    ),
                    'action': (
                        'Comparar los temas negativos del restaurante con '
                        'los indicadores del grupo y actuar primero sobre '
                        'el tema con mayor proporción negativa.'
                    ),
                    'measurement': (
                        'Volver a comparar el rating cuando se acumulen '
                        'nuevas reseñas y comprobar si la brecha disminuye.'
                    ),
                    'score': 60 + abs(reference_group['rating_gap']) * 10
                }
            )
        elif reference_group['rating_gap'] > 0.10:
            strengths.append(
                {
                    'title': 'Rating competitivo',
                    'evidence': (
                        f"Supera al grupo {reference_group['group_label']} "
                        f"por {reference_group['rating_gap']:.2f} puntos."
                    )
                }
            )

        if reference_group['negative_gap'] > 2.00:
            priorities.append(
                {
                    'title': 'Reducir las reseñas negativas',
                    'evidence': (
                        'La proporción negativa supera la referencia de '
                        'sus pares por '
                        f"{reference_group['negative_gap']:.2f} puntos "
                        'porcentuales.'
                    ),
                    'action': (
                        'Revisar primero los temas con mayor porcentaje '
                        'negativo y registrar las acciones aplicadas.'
                    ),
                    'measurement': (
                        'Comparar nuevamente el porcentaje negativo con el '
                        'grupo y comprobar si la diferencia disminuye.'
                    ),
                    'score': 60 + reference_group['negative_gap']
                }
            )
        elif reference_group['negative_gap'] < -2.00:
            strengths.append(
                {
                    'title': 'Menor proporción negativa que sus pares',
                    'evidence': (
                        'Se sitúa '
                        f"{abs(reference_group['negative_gap']):.2f} puntos "
                        'porcentuales por debajo del grupo.'
                    )
                }
            )

    priorities = sorted(
        priorities,
        key=lambda priority: priority['score'],
        reverse=True
    )[:3]

    if not priorities:
        priorities = [
            {
                'title': 'Mantener el seguimiento',
                'evidence': (
                    'No se detectan brechas cuantitativas relevantes con '
                    'el soporte disponible.'
                ),
                'action': (
                    'Continuar revisando las nuevas reseñas y conservar '
                    'los procesos asociados a las fortalezas observadas.'
                ),
                'measurement': (
                    'Actualizar el diagnóstico cuando se acumulen nuevas '
                    'reseñas y confirmar que los indicadores permanecen '
                    'estables.'
                ),
                'score': 0
            }
        ]

    if not strengths:
        strengths = [
            {
                'title': 'Evidencia todavía insuficiente',
                'evidence': (
                    'No existen al menos cinco menciones de un mismo tema '
                    'para identificar una fortaleza temática.'
                )
            }
        ]

    evidence_note = (
        'Los temas se detectan mediante palabras clave en las reseñas. La '
        'polaridad corresponde al sentimiento global de cada reseña. Las '
        'acciones son sugerencias operativas y no resultados causales.'
    )

    return priorities, strengths[:3], evidence_note


def render_deterioration_drivers(restaurant, review_aspects):
    """Explicar las señales asociadas al deterioro."""

    aspects = get_supported_aspects(
        review_aspects,
        restaurant['item_id']
    )
    drivers = get_temporal_drivers(aspects)

    st.markdown(
        '''
        <section class="driver-section">
            <span class="section-eyebrow">SEÑALES DE DETERIORO</span>
            <h2>Qué puede estar impulsando el cambio</h2>
            <p>
                Se comparan los temas negativos de la ventana anterior con
                los de la ventana reciente. El resultado orienta la revisión,
                pero no demuestra causalidad.
            </p>
        </section>
        ''',
        unsafe_allow_html=True
    )

    if not bool(restaurant['alert_evaluable']):
        st.info(
            'No existen al menos cinco reseñas en cada ventana para evaluar '
            'el cambio del restaurante. Esto significa falta de evidencia, '
            'no ausencia de riesgo.'
        )
        return

    if str(restaurant['alert_level']) == 'Sin alerta':
        st.success(
            'Las dos condiciones de deterioro no se cumplen simultáneamente: '
            'caída del rating y aumento de reseñas negativas.'
        )
        return

    if not {
        'temporal_aspect_evaluable',
        'temporal_negative_change'
    }.issubset(aspects.columns):
        st.warning(
            'El archivo temático debe actualizarse para mostrar las señales '
            'temporales. Ejecuta nuevamente la sección 7.7.1.'
        )
        return

    if drivers.empty:
        st.info(
            'El deterioro general está confirmado, pero ningún tema cuenta '
            'con al menos tres menciones en ambas ventanas. Conviene revisar '
            'directamente las reseñas recientes.'
        )
        return

    for row in drivers.head(3).itertuples():
        aspect = escape(str(row.aspect))
        st.markdown(
            f'''
            <article class="driver-card">
                <div>
                    <span class="card-label">TEMA A REVISAR</span>
                    <h3>{aspect}</h3>
                    <p>
                        La proporción negativa aumentó
                        <strong>{row.temporal_negative_change:+.1f} pp</strong>.
                    </p>
                </div>
                <div class="driver-comparison">
                    <div>
                        <span>VENTANA ANTERIOR</span>
                        <strong>{row.previous_negative_percentage:.1f} %</strong>
                        <small>{int(row.previous_mentions)} menciones</small>
                    </div>
                    <div class="driver-arrow">→</div>
                    <div>
                        <span>VENTANA RECIENTE</span>
                        <strong>{row.recent_negative_percentage:.1f} %</strong>
                        <small>{int(row.recent_mentions)} menciones</small>
                    </div>
                </div>
            </article>
            ''',
            unsafe_allow_html=True
        )

    st.caption(
        'Solo se muestran temas con al menos tres menciones en cada ventana. '
        'Un aumento indica asociación temporal, no una causa comprobada.'
    )


def build_actionable_summary(
    restaurant,
    restaurant_groups,
    review_aspects
):
    """Convertir los indicadores en una lectura operativa."""

    negative_percentage = float(restaurant['negative_percentage'])
    summary_parts = [
        f'El {negative_percentage:.1f} % de las reseñas del restaurante es '
        'negativo.'
    ]
    reference_group = get_reference_group(restaurant_groups)

    if reference_group is not None:
        negative_gap = float(reference_group['negative_gap'])
        comparison = 'más' if negative_gap > 0 else 'menos'
        if abs(negative_gap) < 0.05:
            summary_parts.append(
                'Su proporción negativa es prácticamente igual a la de sus '
                f"pares del grupo {reference_group['group_label']}."
            )
        else:
            summary_parts.append(
                f'Tiene {abs(negative_gap):.1f} puntos porcentuales {comparison} '
                'de reseñas negativas que sus pares del grupo '
                f"{reference_group['group_label']}."
            )
    else:
        summary_parts.append(
            'No dispone de un grupo competitivo elegible para calcular una '
            'brecha frente a pares.'
        )

    aspects = get_supported_aspects(
        review_aspects,
        restaurant['item_id']
    )
    drivers = get_temporal_drivers(aspects)
    selected_aspect = None

    if (
        bool(restaurant['alert_evaluable'])
        and str(restaurant['alert_level']) != 'Sin alerta'
        and not drivers.empty
    ):
        driver = drivers.iloc[0]
        selected_aspect = str(driver['aspect'])
        summary_parts.append(
            f'La señal reciente más clara se concentra en '
            f'"{selected_aspect}": su proporción negativa aumentó '
            f"{driver['temporal_negative_change']:.1f} puntos entre ventanas."
        )
    elif not aspects.empty:
        recurring_aspects = (
            aspects.loc[aspects['negative_reviews'].ge(3)]
            .sort_values(
                ['negative_percentage', 'negative_reviews'],
                ascending=False
            )
        )
        if not recurring_aspects.empty:
            recurring = recurring_aspects.iloc[0]
            selected_aspect = str(recurring['aspect'])
            summary_parts.append(
                f'El problema histórico con mayor concentración negativa es '
                f'"{selected_aspect}": '
                f"{recurring['negative_reviews']:.0f} de "
                f"{recurring['mentions']:.0f} menciones son negativas."
            )

    if selected_aspect is not None:
        summary_parts.append(
            'Prioridad sugerida: '
            + aspect_actions[selected_aspect]['action']
        )
    else:
        summary_parts.append(
            'No existe soporte temático suficiente para atribuir el problema '
            'a un área concreta; conviene revisar las reseñas recientes.'
        )

    return ' '.join(summary_parts)


def render_actionable_summary(
    restaurant,
    restaurant_groups,
    review_aspects
):
    """Mostrar la lectura ejecutiva accionable."""

    summary = escape(
        build_actionable_summary(
            restaurant,
            restaurant_groups,
            review_aspects
        )
    )
    st.markdown(
        f'''
        <article class="decision-summary">
            <span class="card-label">LECTURA PARA DECIDIR</span>
            <p>{summary}</p>
        </article>
        ''',
        unsafe_allow_html=True
    )


def render_restaurant_insights(
    restaurant,
    restaurant_groups,
    review_aspects
):
    """Mostrar el plan de mejora antes de las fortalezas."""

    priorities, strengths, evidence_note = build_restaurant_insights(
        restaurant,
        restaurant_groups,
        review_aspects
    )

    render_actionable_summary(
        restaurant,
        restaurant_groups,
        review_aspects
    )

    render_deterioration_drivers(
        restaurant,
        review_aspects
    )

    st.markdown(
        '''
        <section class="action-section">
            <span class="section-eyebrow">PLAN DE ACCIÓN</span>
            <h2>Qué mejorar primero</h2>
            <p>
                Prioridades ordenadas por gravedad, evidencia temática y
                brecha frente a restaurantes similares.
            </p>
        </section>
        ''',
        unsafe_allow_html=True
    )

    for position, priority in enumerate(priorities, start=1):
        title = escape(str(priority['title']))
        evidence = escape(str(priority['evidence']))
        action = escape(str(priority['action']))
        measurement = escape(str(priority['measurement']))
        st.markdown(
            f'''
            <article class="priority-card">
                <div class="priority-card__head">
                    <span class="priority-number">{position:02d}</span>
                    <div>
                        <span class="card-label">PRIORIDAD</span>
                        <h3>{title}</h3>
                    </div>
                </div>
                <div class="priority-evidence">
                    <span class="card-label">EVIDENCIA</span>
                    <p>{evidence}</p>
                </div>
                <div class="priority-grid">
                    <div>
                        <span class="card-label">ACCIÓN SUGERIDA</span>
                        <p>{action}</p>
                    </div>
                    <div>
                        <span class="card-label">CÓMO MEDIRLA</span>
                        <p>{measurement}</p>
                    </div>
                </div>
            </article>
            ''',
            unsafe_allow_html=True
        )

    st.markdown(
        '''
        <section class="strength-section">
            <span class="section-eyebrow">FORTALEZAS</span>
            <h2>Qué conviene conservar</h2>
        </section>
        ''',
        unsafe_allow_html=True
    )
    for strength in strengths:
        title = escape(str(strength['title']))
        evidence = escape(str(strength['evidence']))
        st.markdown(
            f'''
            <article class="strength-card">
                <span class="card-label">FORTALEZA OBSERVADA</span>
                <h3>{title}</h3>
                <p>{evidence}</p>
            </article>
            ''',
            unsafe_allow_html=True
        )

    with st.expander('Cómo se generan las recomendaciones'):
        st.write(evidence_note)
        st.write(
            'Solo se interpretan temas con al menos cinco reseñas que los '
            'mencionan. Los indicadores temporales y competitivos se muestran '
            'únicamente cuando cuentan con el soporte definido en el análisis.'
        )


Overwriting app/insights.py


In [56]:
%%writefile streamlit_app.py

from pathlib import Path

import pandas as pd
import plotly.express as px
import streamlit as st

from app.explainability import explain_sentiment
from app.inference import predict_sentiment
from app.insights import render_restaurant_insights
from app.ui import apply_style, render_header, style_figure
from app.benchmarking import (
    build_executive_summary,
    get_restaurant_groups,
    render_benchmarking,
    render_market_explorer
)


project_path = Path(__file__).resolve().parent
data_path = project_path / 'data'
primary_color = '#05402B'
sentiment_colors = {
    'Negativa': '#8B1E1E',
    'Neutral': '#A8874F',
    'Positiva': primary_color
}

st.set_page_config(
    page_title='Madrid Restaurant Intelligence',
    layout='wide'
)
apply_style()


@st.cache_data
def load_project_data():
    """Cargar los productos analíticos."""

    diagnostics = pd.read_parquet(
        data_path / 'restaurant_diagnostics.parquet'
    )
    membership = pd.read_parquet(
        data_path / 'benchmark_membership.parquet'
    )
    groups = pd.read_parquet(
        data_path / 'benchmark_groups.parquet'
    ).rename(
        columns={
            'restaurants': 'group_restaurants',
            'reviews': 'group_reviews'
        }
    )
    review_aspects = pd.read_parquet(
        data_path / 'restaurant_review_aspects.parquet'
    )

    diagnostics['item_id'] = diagnostics['item_id'].astype(str)
    membership['item_id'] = membership['item_id'].astype(str)
    review_aspects['item_id'] = review_aspects['item_id'].astype(str)

    for data_frame in [membership, groups]:
        data_frame['specialty'] = data_frame['specialty'].astype(str)
        data_frame['price_interval'] = (
            data_frame['price_interval'].astype(str)
        )

    diagnostics['restaurant_label'] = (
        diagnostics['name'].astype(str)
        + ' | '
        + diagnostics['item_id']
    )

    group_metrics = (
        membership
        .merge(
            diagnostics[
                ['item_id', 'mean_review_rating', 'negative_percentage']
            ],
            on='item_id',
            how='left',
            validate='many_to_one'
        )
        .groupby(
            ['specialty', 'price_interval'],
            as_index=False,
            observed=True
        )
        .agg(
            group_mean_rating=('mean_review_rating', 'mean'),
            group_negative_percentage=('negative_percentage', 'mean')
        )
    )
    groups = groups.merge(
        group_metrics,
        on=['specialty', 'price_interval'],
        how='left',
        validate='one_to_one'
    )
    return diagnostics, membership, groups, review_aspects


def format_number(value, decimals=2, suffix=''):
    """Formatear un indicador."""

    if pd.isna(value):
        return 'No disponible'
    return f'{float(value):,.{decimals}f}{suffix}'


try:
    diagnostics, membership, groups, review_aspects = load_project_data()
except Exception as error:
    st.error(f'No fue posible cargar los datos: {error}')
    st.stop()

active_levels = ['Vigilancia', 'Alerta alta', 'Alerta crítica']
alert_names = {
    'Sin alerta': 'Estable',
    'Vigilancia': 'Vigilancia',
    'Alerta alta': 'Deterioro alto',
    'Alerta crítica': 'Deterioro severo',
    'No evaluable': 'Sin datos temporales suficientes'
}
render_header()

# Navegar entre las dos vistas principales
if 'active_page' not in st.session_state:
    st.session_state['active_page'] = 'restaurant'

st.sidebar.markdown('### Navegación')
if st.sidebar.button(
    'Diagnóstico del restaurante',
    use_container_width=True,
    type=(
        'primary'
        if st.session_state['active_page'] == 'restaurant'
        else 'secondary'
    )
):
    st.session_state['active_page'] = 'restaurant'

if st.sidebar.button(
    'Explorar competencia · Top 10',
    use_container_width=True,
    type=(
        'primary'
        if st.session_state['active_page'] == 'market'
        else 'secondary'
    )
):
    st.session_state['active_page'] = 'market'

st.sidebar.divider()

if st.session_state['active_page'] == 'market':
    render_market_explorer(
        groups,
        membership,
        diagnostics
    )
    st.divider()
    st.caption('TFM | Ciencia de Datos, Big Data y Business Analytics')
    st.stop()

# Indicadores generales
evaluable_count = int(diagnostics['alert_evaluable'].sum())
temporal_coverage = evaluable_count / len(diagnostics) * 100
overview = st.columns(4)
overview[0].metric('Restaurantes', f"{diagnostics['item_id'].nunique():,}")
overview[1].metric('Reseñas', f"{diagnostics['reviews'].sum():,}")
overview[2].metric('Grupos competitivos', f'{len(groups):,}')
overview[3].metric(
    'Cobertura temporal',
    f'{evaluable_count:,} ({temporal_coverage:.1f} %)'
)

with st.expander('Cómo interpretar toda la aplicación'):
    st.markdown(
        '- **Diagnóstico:** resume el historial completo de reseñas.\n'
        '- **Pares:** comparten especialidad y rango de precio.\n'
        '- **Brecha de rating:** positiva significa mejor desempeño.\n'
        '- **Brecha de negativas:** positiva significa peor desempeño.\n'
        '- **Monitoreo temporal:** compara dos ventanas consecutivas.\n'
        '- **Sin datos temporales suficientes:** no es una alerta.\n'
        '- **Temas de reseñas:** requieren al menos cinco menciones.\n'
        '- **Modelo de texto:** su puntuación normalizada no es una '
        'probabilidad calibrada.'
    )

# Restaurante seleccionado
restaurant_search = st.sidebar.text_input(
    'Buscar restaurante',
    placeholder='Escribe el nombre'
).strip()

if restaurant_search:
    filtered_restaurants = diagnostics.loc[
        diagnostics['name'].astype(str).str.contains(
            restaurant_search,
            case=False,
            na=False,
            regex=False
        )
    ]
else:
    filtered_restaurants = diagnostics

if filtered_restaurants.empty:
    st.sidebar.warning('No se encontraron restaurantes con ese nombre.')
    filtered_restaurants = diagnostics

restaurant_options = (
    filtered_restaurants
    .sort_values(['name', 'item_id'])['restaurant_label']
    .tolist()
)
selected_label = st.sidebar.selectbox('Resultados', restaurant_options)
restaurant = diagnostics.loc[
    diagnostics['restaurant_label'].eq(selected_label)
].iloc[0]

restaurant_groups = get_restaurant_groups(
    restaurant,
    membership,
    groups
)

(
    diagnosis_tab,
    benchmark_tab,
    alerts_tab,
    review_tab,
    methodology_tab
) = st.tabs(
    [
        'Diagnóstico',
        'Comparación con pares',
        'Alertas del mercado',
        'Analizar reseña',
        'Cómo se calcula'
    ]
)


with diagnosis_tab:
    st.header(str(restaurant['name']))

    metrics = st.columns(4)
    metrics[0].metric('Reseñas', f"{int(restaurant['reviews']):,}")
    metrics[1].metric(
        'Rating medio',
        format_number(restaurant['mean_review_rating'])
    )
    metrics[2].metric(
        'Positivas',
        format_number(restaurant['positive_percentage'], suffix='%')
    )
    metrics[3].metric(
        'Negativas',
        format_number(restaurant['negative_percentage'], suffix='%')
    )
    st.caption(
        f"Soporte: {restaurant['diagnostic_support']} | "
        f"Estado: {restaurant['diagnostic_status']}"
    )

    with st.expander('Qué significan estos indicadores'):
        st.markdown(
            '- **Reseñas:** volumen histórico disponible.\n'
            '- **Rating medio:** promedio de valoraciones entre 1 y 5.\n'
            '- **Positivas y negativas:** distribución del sentimiento '
            'derivada de la valoración.\n'
            '- **Soporte:** estabilidad esperada según el número de reseñas.'
        )

    st.subheader('Lectura ejecutiva')
    st.info(
        build_executive_summary(
            restaurant,
            restaurant_groups
        )
    )

    alert_metrics = st.columns(4)
    displayed_alert = alert_names.get(
        str(restaurant['alert_level']),
        str(restaurant['alert_level'])
    )
    alert_metrics[0].metric('Estado temporal', displayed_alert)
    alert_metrics[1].metric(
        'Acción recomendada',
        str(restaurant['operational_priority'])
    )
    alert_metrics[2].metric(
        'Cambio rating',
        format_number(restaurant['rating_change'])
    )
    alert_metrics[3].metric(
        'Cambio negativas',
        format_number(
            restaurant['negative_percentage_point_change'],
            suffix=' pp'
        )
    )

    st.caption(
        'El cambio compara la ventana reciente con la anterior. Un descenso '
        'del rating y un aumento de negativas indican deterioro.'
    )

    render_restaurant_insights(
        restaurant,
        restaurant_groups,
        review_aspects
    )

    with st.expander('Ver distribución histórica del sentimiento'):
        sentiment_data = pd.DataFrame(
            {
                'Sentimiento': ['Negativa', 'Neutral', 'Positiva'],
                'Porcentaje': [
                    restaurant['negative_percentage'],
                    restaurant['neutral_percentage'],
                    restaurant['positive_percentage']
                ]
            }
        )
        sentiment_figure = px.bar(
            sentiment_data,
            x='Sentimiento',
            y='Porcentaje',
            color='Sentimiento',
            color_discrete_map=sentiment_colors
        )
        sentiment_figure.update_layout(showlegend=False)
        st.plotly_chart(
            style_figure(sentiment_figure),
            use_container_width=True
        )
        st.caption(
            'La gráfica muestra la distribución histórica, no únicamente '
            'el periodo reciente.'
        )


with benchmark_tab:
    render_benchmarking(
        restaurant,
        restaurant_groups,
        membership,
        diagnostics
    )


with alerts_tab:
    st.header('Monitoreo temporal de la reputación')

    st.info(
        'Solo se evalúan restaurantes con al menos cinco reseñas en cada '
        'ventana. Los demás no tienen una alerta: carecen de soporte temporal.'
    )

    evaluable = diagnostics.loc[diagnostics['alert_evaluable']]
    signals = evaluable.loc[evaluable['alert_level'].isin(active_levels)]
    stable = evaluable['alert_level'].eq('Sin alerta').sum()

    alert_overview = st.columns(4)
    alert_overview[0].metric(
        'Cobertura',
        f'{len(evaluable):,} ({len(evaluable) / len(diagnostics) * 100:.1f} %)'
    )
    alert_overview[1].metric('Estables', f'{stable:,}')
    alert_overview[2].metric('Con señal', f'{len(signals):,}')
    alert_overview[3].metric(
        'Deterioro severo',
        f"{evaluable['alert_level'].eq('Alerta crítica').sum():,}"
    )

    with st.expander('Qué significa cada nivel'):
        st.markdown(
            '- **Vigilancia:** rating −0.40 o más y negativas +10 puntos '
            'porcentuales o más.\n'
            '- **Deterioro alto:** rating −0.80 o más y negativas +20 puntos.\n'
            '- **Deterioro severo:** rating −1.00 o más y negativas +30 puntos.\n'
            '- Las dos condiciones deben cumplirse. La señal describe un '
            'cambio observado y no predice el futuro.'
        )

    selected_levels = st.multiselect(
        'Filtrar señales',
        active_levels,
        default=['Alerta alta', 'Alerta crítica'],
        format_func=lambda level: alert_names[level]
    )
    alert_table = (
        signals.loc[
            signals['alert_level'].isin(selected_levels),
            [
                'name',
                'price_interval',
                'recent_reviews',
                'rating_change',
                'negative_percentage_point_change',
                'alert_level',
                'operational_priority'
            ]
        ]
        .sort_values('rating_change')
        .rename(
            columns={
                'name': 'Restaurante',
                'price_interval': 'Precio',
                'recent_reviews': 'Reseñas recientes',
                'rating_change': 'Cambio rating',
                'negative_percentage_point_change': 'Cambio negativas (pp)',
                'alert_level': 'Alerta',
                'operational_priority': 'Prioridad'
            }
        )
    )
    alert_table['Alerta'] = alert_table['Alerta'].map(alert_names)
    st.dataframe(
        alert_table.round(2),
        hide_index=True,
        use_container_width=True
    )


with review_tab:
    st.header('Clasificación explicable de una reseña')

    with st.expander('Qué analiza el modelo'):
        st.markdown(
            '- Clasifica el texto como **Negativa**, **Neutral** o '
            '**Positiva**.\n'
            '- La puntuación normalizada facilita la comparación entre '
            'clases, pero no es una probabilidad.\n'
            '- LIME muestra qué términos apoyan o contradicen la clase '
            'predicha.\n'
            '- La explicación corresponde únicamente a la reseña escrita.'
        )

    review_text = st.text_area(
        'Texto de la reseña',
        value='La comida estaba fría y el servicio fue demasiado lento.',
        height=140,
        max_chars=5_000
    )

    if st.button('Analizar reseña', type='primary'):
        try:
            prediction = predict_sentiment(review_text)
            explanation = explain_sentiment(review_text)
        except Exception as error:
            st.error(f'No fue posible analizar la reseña: {error}')
        else:
            sentiment = prediction['sentiment']
            message_method = {
                'Negativa': st.error,
                'Neutral': st.warning,
                'Positiva': st.success
            }[sentiment]
            message_method(f'Sentimiento: {sentiment}')

            explanation_metrics = st.columns(3)
            explanation_metrics[0].metric(
                'Puntuación normalizada',
                format_number(explanation['normalized_class_score'], 4)
            )
            explanation_metrics[1].metric(
                'Fidelidad local R²',
                format_number(explanation['local_fidelity_r2'], 4)
            )
            explanation_metrics[2].metric(
                'Inferencia',
                format_number(
                    prediction['prediction_milliseconds'],
                    2,
                    ' ms'
                )
            )

            contribution_rows = explanation.get(
                'contributions',
                explanation.get('terms', [])
            )
            contributions = pd.DataFrame(contribution_rows)
            contributions['Dirección'] = (
                contributions['contribution']
                .gt(0)
                .map({True: 'Apoya', False: 'Contradice'})
            )
            lime_figure = px.bar(
                contributions.sort_values('contribution'),
                x='contribution',
                y='term',
                color='Dirección',
                orientation='h',
                color_discrete_map={
                    'Apoya': primary_color,
                    'Contradice': '#A8B6AE'
                },
                labels={
                    'contribution': 'Contribución',
                    'term': 'Término'
                }
            )
            st.plotly_chart(
                style_figure(lime_figure),
                use_container_width=True
            )
            st.caption(
                'La puntuación normalizada no es una probabilidad calibrada.'
            )


with methodology_tab:
    st.header('Cómo se calcula cada resultado')
    st.info(
        'Esta sección explica el origen, el cálculo y los límites de todos '
        'los indicadores de la aplicación.'
    )

    with st.expander('1. Datos utilizados', expanded=True):
        st.markdown(
            f'- Se utilizan **{diagnostics["reviews"].sum():,} reseñas** de '
            f'**{diagnostics["item_id"].nunique():,} restaurantes** de Madrid.\n'
            '- Se conservaron reseñas en español e inglés.\n'
            '- Los duplicados y registros fuera de las reglas de calidad se '
            'corrigieron en el Notebook 1.\n'
            '- El Notebook 2 construyó el diagnóstico, los grupos competitivos '
            'y las ventanas temporales.\n'
            '- El Notebook 3 entrenó el modelo de sentimiento y construyó '
            'esta aplicación.'
        )

    with st.expander('2. Diagnóstico histórico'):
        st.markdown(
            '- **Rating medio:** promedio de las valoraciones de todas las '
            'reseñas del restaurante, expresado entre 1 y 5.\n'
            '- **Negativa:** valoración 1 o 2.\n'
            '- **Neutral:** valoración 3.\n'
            '- **Positiva:** valoración 4 o 5.\n'
            '- **Soporte muy bajo:** 1–9 reseñas; **bajo:** 10–29; '
            '**adecuado:** 30–99; **alto:** 100 o más.\n'
            '- El soporte indica cuánta evidencia respalda el diagnóstico; '
            'no califica la calidad del restaurante.'
        )

    with st.expander('3. Comparación con pares'):
        st.markdown(
            '- Un par comparte **especialidad** y **rango de precio**.\n'
            '- Un grupo competitivo se conserva si contiene al menos '
            '**30 restaurantes y 1,000 reseñas**.\n'
            '- **Brecha de rating = rating del restaurante − rating del grupo.** '
            'Un valor positivo es favorable.\n'
            '- **Brecha de negativas = % negativo del restaurante − % negativo '
            'del grupo.** Un valor positivo es desfavorable.\n'
            '- Los promedios del grupo se calculan a nivel de restaurante para '
            'que el volumen de un solo establecimiento no domine el resultado.'
        )

    with st.expander('4. Alertas de deterioro'):
        st.markdown(
            '- Se excluye el último mes observado porque puede estar incompleto.\n'
            '- Se comparan dos ventanas consecutivas de **seis meses completos**.\n'
            '- Un restaurante es evaluable si tiene al menos **cinco reseñas '
            'en cada ventana**.\n'
            '- **Cambio de rating = rating reciente − rating anterior.**\n'
            '- **Cambio de negativas = % negativo reciente − % negativo '
            'anterior.**\n'
            '- **Vigilancia:** rating ≤ −0.40 y negativas ≥ +10 puntos.\n'
            '- **Deterioro alto:** rating ≤ −0.80 y negativas ≥ +20 puntos.\n'
            '- **Deterioro severo:** rating ≤ −1.00 y negativas ≥ +30 puntos.\n'
            '- Las dos condiciones deben cumplirse simultáneamente.\n'
            '- **Sin datos temporales suficientes no es una alerta:** significa '
            'que no existe evidencia suficiente para evaluar el cambio.\n'
            '- La alerta describe un deterioro observado; no predice que '
            'continuará en el futuro.'
        )

    with st.expander('5. Temas asociados al deterioro'):
        st.markdown(
            '- Se buscan expresiones en español e inglés relacionadas con '
            'comida, servicio, ambiente, precio, espera y limpieza.\n'
            '- Para mostrar un tema temporal se exigen al menos **tres '
            'menciones en cada ventana**.\n'
            '- Se compara el porcentaje de reseñas negativas que menciona '
            'cada tema entre ambas ventanas.\n'
            '- Un aumento ayuda a identificar qué revisar, pero representa '
            'una **asociación temporal y no una causa comprobada**.\n'
            '- Si no existe soporte temático, la aplicación recomienda revisar '
            'directamente las reseñas recientes.'
        )

    with st.expander('6. Recomendaciones operativas'):
        st.markdown(
            '- Se combinan cuatro evidencias: gravedad de la alerta, cambio '
            'temático, problemas históricos y brecha frente a pares.\n'
            '- Las evidencias se ordenan y se muestran hasta tres prioridades.\n'
            '- Cada prioridad incluye el dato observado, una acción sugerida '
            'y la forma de comprobar si mejora.\n'
            '- Las acciones son reglas transparentes de apoyo a la decisión; '
            'deben validarse con la operación real del restaurante.'
        )

    with st.expander('7. Modelo de sentimiento y LIME'):
        st.markdown(
            '- El modelo final es **TF-IDF + LinearSVC**, seleccionado mediante '
            'validación y evaluado una sola vez sobre prueba.\n'
            '- En prueba obtuvo `accuracy=0.9048`, `macro_f1=0.8078` y '
            '`negative_recall=0.8793`.\n'
            '- LIME modifica localmente el texto para estimar qué palabras '
            'apoyan o contradicen la predicción.\n'
            '- La puntuación normalizada facilita la lectura, pero no es una '
            'probabilidad calibrada ni explica el restaurante completo.'
        )


st.divider()
st.caption('TFM | Ciencia de Datos, Big Data y Business Analytics')


Overwriting streamlit_app.py


### 7.7.3. Validación de la aplicación

Se comprueba la existencia y la sintaxis de los cuatro archivos de la
interfaz antes de iniciar Streamlit.


In [57]:


# Definir los archivos de la aplicación
application_files = {
    'visual_interface': project_path / 'app' / 'ui.py',
    'benchmarking': project_path / 'app' / 'benchmarking.py',
    'review_insights': project_path / 'app' / 'insights.py',
    'streamlit_application': project_path / 'streamlit_app.py'
}


def has_valid_syntax(file_path):
    # Validar la sintaxis sin ejecutar el archivo
    if not file_path.exists():
        return False
    try:
        ast.parse(file_path.read_text(encoding='utf-8'))
    except SyntaxError:
        return False
    return True


application_validation = pd.DataFrame(
    [
        {
            'file': file_path.name,
            'exists': file_path.exists(),
            'valid_syntax': has_valid_syntax(file_path)
        }
        for file_path in application_files.values()
    ],
    index=application_files
)

display(application_validation)

assert application_validation['exists'].all()
assert application_validation['valid_syntax'].all()


,file,exists,valid_syntax
visual_interface,ui.py,True,True
benchmarking,benchmarking.py,True,True
review_insights,insights.py,True,True
streamlit_application,streamlit_app.py,True,True


### 7.7.4. Ejecución local de la aplicación

La ejecución previa se cierra de forma segura y Streamlit se inicia como
un proceso local. Después se comprueba su disponibilidad antes de abrir la
interfaz.


In [58]:


# Cerrar el proceso conservado por el notebook
if 'streamlit_process' in globals():
    if streamlit_process.poll() is None:
        streamlit_process.terminate()
        streamlit_process.wait(timeout=10)

if 'streamlit_log' in globals():
    if not streamlit_log.closed:
        streamlit_log.close()


# Cerrar cualquier servidor anterior en el puerto 8501
port_result = subprocess.run(
    [
        'lsof',
        '-tiTCP:8501',
        '-sTCP:LISTEN'
    ],
    capture_output=True,
    text=True,
    check=False
)

port_processes = {
    int(process_id)
    for process_id in port_result.stdout.split()
    if process_id.isdigit()
}

for process_id in port_processes:
    if process_id != os.getpid():
        os.kill(process_id, signal.SIGTERM)

if port_processes:
    sleep(2)

print('Servidor anterior cerrado')


Servidor anterior cerrado


In [59]:



streamlit_app_path = (
    project_path
    / 'streamlit_app.py'
)

streamlit_log_path = (
    project_path
    / 'artifacts'
    / 'streamlit_app.log'
)

streamlit_log_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

streamlit_log = open(
    streamlit_log_path,
    mode='w',
    encoding='utf-8'
)

streamlit_process = subprocess.Popen(
    [
        sys.executable,
        '-m',
        'streamlit',
        'run',
        str(streamlit_app_path),
        '--server.headless=true',
        '--server.address=localhost',
        '--server.port=8501',
        '--server.fileWatcherType=auto',
        '--browser.gatherUsageStats=false'
    ],
    cwd=project_path,
    stdout=streamlit_log,
    stderr=subprocess.STDOUT
)

streamlit_url = (
    'http://localhost:8501'
)

application_available = False

for _ in range(30):

    try:

        with urlopen(
            streamlit_url
            + '/_stcore/health',
            timeout=1
        ) as response:

            application_available = (
                response.status == 200
            )

        if application_available:
            break

    except Exception:
        sleep(1)


execution_summary = pd.Series(
    {
        'process_id': (
            streamlit_process.pid
        ),
        'application_available': (
            application_available
        ),
        'url': streamlit_url,
        'log_file': (
            streamlit_log_path.name
        )
    },
    name='result'
)

display(
    execution_summary
    .to_frame()
)

if application_available:

    display(
        HTML(
            f'''
            <a
                href="{streamlit_url}"
                target="_blank"
                style="
                    display:inline-block;
                    padding:12px 22px;
                    color:white;
                    background:#05402B;
                    border-radius:20px;
                    text-decoration:none;
                    font-weight:600;
                "
            >
                Abrir aplicación
            </a>
            '''
        )
    )

else:

    streamlit_log.flush()

    print(
        streamlit_log_path.read_text(
            encoding='utf-8'
        )
    )


,result
process_id,8160
application_available,True
url,http://localhost:8501
log_file,streamlit_app.log


In [60]:
# Mostrar los errores de ejecución
streamlit_log.flush()

log_text = streamlit_log_path.read_text(
    encoding='utf-8',
    errors='replace'
)

log_lines = log_text.splitlines()

print(
    '\n'.join(
        log_lines[-120:]
    )
)


  You can now view your Streamlit app in your browser.

  URL: http://0.0.0.0:10000



## 7.8. Preparación para Streamlit Cloud

Se generan las dependencias y la configuración mínima de despliegue.


In [61]:
requirements = '''streamlit
pandas
numpy
plotly
scikit-learn
joblib
lime
pyarrow
fastapi
pydantic
'''
(project_path / 'requirements.txt').write_text(requirements, encoding='utf-8')
print('requirements.txt creado')


requirements.txt creado


In [62]:
streamlit_config_path = project_path / '.streamlit' / 'config.toml'
streamlit_config_path.parent.mkdir(parents=True, exist_ok=True)
streamlit_config_path.write_text(
    '[server]\nheadless = true\n\n[browser]\ngatherUsageStats = false\n',
    encoding='utf-8'
)
print('.streamlit/config.toml creado')


.streamlit/config.toml creado


### 7.8.1. Archivos para el despliegue

Subir a GitHub `streamlit_app.py`, `app/`, `artifacts/models/`, `data/`, `requirements.txt` y `.streamlit/config.toml`. El archivo de entrada es `streamlit_app.py`.
